# Heat Exchanger Analysis — Complete Engineering Suite (Upgraded)

**Shell-and-Tube · Plate · Finned-Tube · Transient Dynamics · Fouling (Kern–Seaton) · ML Prediction · Uncertainty · Sensitivity · Optimization**

| Module | Capability |
|--------|------------|
| Core HX Models | LMTD, ε-NTU, overall U, pressure drop |
| Transient | First-order + distributed lag response |
| Fouling | Kern–Seaton asymptotic model + synthetic data generation |
| ML | Random Forest fouling resistance predictor |
| Uncertainty | Monte-Carlo propagation on U, Q, effectiveness |
| Sensitivity | Local + global (Sobol-style) parameter ranking |
| Optimization | Maximize Q or minimize cost under constraints |

> **Upgrade notes (v2)**  
> • Fully self-contained (all classes defined)  
> • Clean modular OOP design  
> • Reproducible random seeds  
> • Modern visualizations + summary dashboard  
> • Predictive Maintenance · Fleet · Nanofluids · Economics · CSV Ingestion · PDF Reports · Ready for CFD / digital-twin extension


## 1. Imports & Global Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize, differential_evolution
from scipy.integrate import odeint
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

# Plot style
plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.titleweight': 'bold'
})
sns.set_palette('husl')

print("Libraries loaded successfully.")

## 2. Core Heat Exchanger Classes

Three industrial configurations are implemented with a common interface.

### Input validation

All core constructors and `steady_state` / `transient_response` methods now raise a clear `ValidationError` for:

- Non-positive flows, areas, U, lengths, tube counts
- `tube_id >= tube_od` or `fin diameter <= tube od`
- Hot inlet colder than cold inlet (temperature cross at the boundary)
- Invalid `flow_type`, empty / too-small fouling dataframes, bad `test_size`

This prevents silent NaNs and makes debugging plant-data issues much easier.


In [ ]:
class ValidationError(ValueError):
    """Raised when heat-exchanger inputs are physically invalid."""
    pass


def _check_positive(name, value, allow_zero=False):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        raise ValidationError(f"{name} must be a number, got {value!r}")
    if allow_zero:
        if value < 0:
            raise ValidationError(f"{name} must be >= 0, got {value}")
    else:
        if value <= 0:
            raise ValidationError(f"{name} must be > 0, got {value}")
    return value


def _check_temperature_pair(Th_in, Tc_in, context=""):
    _check_positive(f"{context}Th_in", Th_in, allow_zero=True)
    _check_positive(f"{context}Tc_in", Tc_in, allow_zero=True)
    if Th_in <= Tc_in:
        raise ValidationError(
            f"Hot inlet must be warmer than cold inlet (Th_in={Th_in}, Tc_in={Tc_in}). "
            "Check stream assignment or units."
        )


class BaseHeatExchanger:
    """Common thermal calculations (LMTD / ε-NTU) with input validation."""

    def __init__(self, name="HX"):
        self.name = str(name) if name else "HX"

    @staticmethod
    def lmtd(dT1, dT2):
        """Log-mean temperature difference (handles equal ΔT)."""
        if dT1 is None or dT2 is None or np.isnan(dT1) or np.isnan(dT2):
            return np.nan
        if dT1 <= 0 or dT2 <= 0:
            # Temperature cross or invalid driving force
            return np.nan
        if abs(dT1 - dT2) < 1e-6:
            return float(dT1)
        return float((dT1 - dT2) / np.log(dT1 / dT2))

    def effectiveness_ntu(self, NTU, Cr, flow_type='counter'):
        """ε-NTU relations for common flow arrangements."""
        NTU = float(NTU)
        Cr = float(Cr)
        if NTU < 0:
            raise ValidationError(f"NTU must be >= 0, got {NTU}")
        if Cr < 0 or Cr > 1.0001:
            raise ValidationError(f"Capacity ratio Cr must be in [0, 1], got {Cr}")
        Cr = min(Cr, 1.0)

        flow_type = str(flow_type).lower().strip()
        if flow_type not in ('counter', 'parallel', 'cross'):
            raise ValidationError("flow_type must be 'counter' | 'parallel' | 'cross'")

        if NTU == 0:
            return 0.0
        if Cr >= 0.999:                       # balanced counter-flow limit
            return NTU / (1 + NTU)
        if flow_type == 'counter':
            return (1 - np.exp(-NTU * (1 - Cr))) / (1 - Cr * np.exp(-NTU * (1 - Cr)))
        elif flow_type == 'parallel':
            return (1 - np.exp(-NTU * (1 + Cr))) / (1 + Cr)
        else:  # cross – unmixed-unmixed approximation
            return 1 - np.exp((1 / Cr) * NTU ** 0.22 * (np.exp(-Cr * NTU ** 0.78) - 1))

    def heat_duty(self, m_h, cp_h, Th_in, Th_out):
        _check_positive("m_h", m_h)
        _check_positive("cp_h", cp_h)
        return m_h * cp_h * (Th_in - Th_out)


class ShellTubeHeatExchanger(BaseHeatExchanger):
    """1-shell-pass, multi-tube-pass shell-and-tube HX."""

    def __init__(self, n_tubes=100, tube_length=3.0, tube_od=0.025, tube_id=0.020,
                 shell_id=0.4, baffle_spacing=0.3, n_baffles=8,
                 k_wall=15.0, name="Shell-Tube HX"):
        super().__init__(name)
        self.n_tubes = int(_check_positive("n_tubes", n_tubes))
        self.L = float(_check_positive("tube_length", tube_length))
        self.do = float(_check_positive("tube_od", tube_od))
        self.di = float(_check_positive("tube_id", tube_id))
        if self.di >= self.do:
            raise ValidationError(f"tube_id ({self.di}) must be < tube_od ({self.do})")
        self.Ds = float(_check_positive("shell_id", shell_id))
        if self.Ds <= self.do:
            raise ValidationError(f"shell_id ({self.Ds}) must be > tube_od ({self.do})")
        self.B = float(_check_positive("baffle_spacing", baffle_spacing))
        self.n_baffles = int(_check_positive("n_baffles", n_baffles, allow_zero=True))
        self.k_wall = float(_check_positive("k_wall", k_wall))

        # Derived geometry
        self.A_tube_i = np.pi * self.di * self.L * self.n_tubes
        self.A_tube_o = np.pi * self.do * self.L * self.n_tubes
        self.A = self.A_tube_o

    def overall_U(self, hi, ho, Rf_i=0.0, Rf_o=0.0):
        """Overall heat-transfer coefficient referred to outer surface."""
        _check_positive("hi", hi)
        _check_positive("ho", ho)
        Rf_i = max(float(Rf_i), 0.0)
        Rf_o = max(float(Rf_o), 0.0)
        Rw = (self.do / (2 * self.k_wall)) * np.log(self.do / self.di)
        U = 1.0 / (
            (self.do / self.di) * (1 / hi + Rf_i) +
            Rw +
            1 / ho + Rf_o
        )
        if U <= 0 or not np.isfinite(U):
            raise ValidationError(f"Computed U is invalid: {U}")
        return float(U)

    def steady_state(self, m_h, cp_h, Th_in, m_c, cp_c, Tc_in,
                     U=800.0, flow_type='counter'):
        """ε-NTU steady-state solution with validation."""
        _check_positive("m_h", m_h)
        _check_positive("cp_h", cp_h)
        _check_positive("m_c", m_c)
        _check_positive("cp_c", cp_c)
        _check_positive("U", U)
        _check_temperature_pair(Th_in, Tc_in)

        C_h = m_h * cp_h
        C_c = m_c * cp_c
        C_min = min(C_h, C_c)
        C_max = max(C_h, C_c)
        Cr = C_min / C_max
        NTU = U * self.A / C_min
        eps = self.effectiveness_ntu(NTU, Cr, flow_type)
        Q = eps * C_min * (Th_in - Tc_in)
        Th_out = Th_in - Q / C_h
        Tc_out = Tc_in + Q / C_c

        # Soft check for temperature cross (should not happen with valid ε-NTU)
        if Th_out < Tc_out - 0.5:
            import warnings
            warnings.warn(
                f"Possible temperature cross: Th_out={Th_out:.2f}, Tc_out={Tc_out:.2f}. "
                "Check inputs or flow arrangement.",
                RuntimeWarning
            )

        return {
            'Q': Q, 'eps': eps, 'NTU': NTU, 'Cr': Cr,
            'Th_out': Th_out, 'Tc_out': Tc_out, 'U': U, 'A': self.A
        }

    def transient_response(self, m_h, cp_h, m_c, cp_c, U,
                           Th_in_step=(100, 140), Tc_in=20.0,
                           t_step=60.0, t_final=600.0, n_points=600):
        """
        Lumped first-order transient model for hot-fluid outlet temperature.
        Step change in hot inlet temperature at t = t_step.
        """
        _check_positive("m_h", m_h)
        _check_positive("cp_h", cp_h)
        _check_positive("m_c", m_c)
        _check_positive("cp_c", cp_c)
        _check_positive("U", U)
        if t_final <= t_step:
            raise ValidationError(f"t_final ({t_final}) must be > t_step ({t_step})")
        if n_points < 10:
            raise ValidationError("n_points must be >= 10")
        if len(Th_in_step) != 2:
            raise ValidationError("Th_in_step must be a (T_before, T_after) pair")

        C_h = m_h * cp_h
        C_c = m_c * cp_c
        C_min = min(C_h, C_c)
        M_cp = 1.8e6
        tau = M_cp / (U * self.A)

        t = np.linspace(0, t_final, int(n_points))
        Th_in = np.where(t < t_step, Th_in_step[0], Th_in_step[1])

        def ss_out(Thin):
            res = self.steady_state(m_h, cp_h, Thin, m_c, cp_c, Tc_in, U)
            return res['Th_out']

        Th_out_ss0 = ss_out(Th_in_step[0])
        Th_out_ss1 = ss_out(Th_in_step[1])

        Th_out = np.zeros_like(t)
        for i, ti in enumerate(t):
            if ti < t_step:
                Th_out[i] = Th_out_ss0
            else:
                dt = ti - t_step
                Th_out[i] = Th_out_ss1 + (Th_out_ss0 - Th_out_ss1) * np.exp(-dt / tau)

        return t, Th_out, Th_in, tau

    def generate_fouling_data(self, n_samples=2000):
        """Synthetic operational data using asymptotic Kern–Seaton model."""
        n_samples = int(_check_positive("n_samples", n_samples))
        if n_samples < 50:
            raise ValidationError("n_samples should be at least 50 for useful ML data")
        hours = np.random.uniform(0, 40000, n_samples)
        m_hot = np.random.uniform(1.5, 4.0, n_samples)
        T_hot = np.random.uniform(80, 180, n_samples)
        hardness = np.random.uniform(50, 600, n_samples)
        pH = np.random.uniform(6.5, 8.5, n_samples)
        velocity = np.random.uniform(0.5, 2.5, n_samples)

        Rf_inf = 1.2e-4 * (hardness / 200) ** 1.1 * np.exp(0.012 * (T_hot - 100))
        tau_f = 8000 * (velocity / 1.5) ** 0.8 / (hardness / 200) ** 0.3
        Rf = Rf_inf * (1 - np.exp(-hours / tau_f))
        Rf += np.random.normal(0, 5e-6, n_samples)
        Rf = np.clip(Rf, 0, None)

        return pd.DataFrame({
            'operating_hours': hours,
            'm_dot_hot': m_hot,
            'T_hot_in': T_hot,
            'water_hardness': hardness,
            'pH': pH,
            'velocity': velocity,
            'Rf': Rf
        })

    def train_fouling_ml_model(self, df, test_size=0.2):
        if df is None or len(df) < 20:
            raise ValidationError("Fouling dataframe must contain at least 20 rows")
        features = ['operating_hours', 'm_dot_hot', 'T_hot_in',
                    'water_hardness', 'pH', 'velocity']
        missing = [f for f in features + ['Rf'] if f not in df.columns]
        if missing:
            raise ValidationError(f"Dataframe missing columns: {missing}")
        if not (0.05 <= test_size <= 0.5):
            raise ValidationError("test_size should be between 0.05 and 0.5")

        X = df[features]
        y = df['Rf']
        if y.isna().any() or (y < 0).any():
            raise ValidationError("Rf column contains NaN or negative values")

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42)

        model = RandomForestRegressor(
            n_estimators=200, max_depth=12, min_samples_leaf=3,
            random_state=42, n_jobs=-1)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        return model, r2, mae, X_test, y_test, y_pred


class PlateHeatExchanger(BaseHeatExchanger):
    """Gasketed plate heat exchanger (simplified)."""

    def __init__(self, n_plates=50, plate_area=0.3, gap=0.003, name="Plate HX"):
        super().__init__(name)
        self.n_plates = int(_check_positive("n_plates", n_plates))
        self.A_plate = float(_check_positive("plate_area", plate_area))
        self.gap = float(_check_positive("gap", gap))
        self.A = self.n_plates * self.A_plate * 0.95

    def overall_U(self, hi=5000, ho=5000, Rf=1e-5):
        _check_positive("hi", hi)
        _check_positive("ho", ho)
        Rf = max(float(Rf), 0.0)
        U = 1.0 / (1/hi + 1/ho + 2*Rf)
        if not np.isfinite(U) or U <= 0:
            raise ValidationError(f"Computed U is invalid: {U}")
        return float(U)

    def steady_state(self, m_h, cp_h, Th_in, m_c, cp_c, Tc_in, U=None):
        _check_positive("m_h", m_h)
        _check_positive("cp_h", cp_h)
        _check_positive("m_c", m_c)
        _check_positive("cp_c", cp_c)
        _check_temperature_pair(Th_in, Tc_in)
        if U is None:
            U = self.overall_U()
        else:
            _check_positive("U", U)
        C_h, C_c = m_h * cp_h, m_c * cp_c
        C_min = min(C_h, C_c)
        Cr = C_min / max(C_h, C_c)
        NTU = U * self.A / C_min
        eps = self.effectiveness_ntu(NTU, Cr, 'counter')
        Q = eps * C_min * (Th_in - Tc_in)
        return {
            'Q': Q, 'eps': eps, 'NTU': NTU,
            'Th_out': Th_in - Q / C_h,
            'Tc_out': Tc_in + Q / C_c, 'U': U, 'A': self.A
        }


class FinnedTubeHeatExchanger(BaseHeatExchanger):
    """Air-cooled finned-tube HX (annular fins)."""

    def __init__(self, n_tubes=80, L=4.0, do=0.025, df=0.045,
                 n_fins_per_m=300, fin_thickness=0.0004, name="Finned-Tube HX"):
        super().__init__(name)
        self.n_tubes = int(_check_positive("n_tubes", n_tubes))
        self.L = float(_check_positive("L", L))
        self.do = float(_check_positive("do", do))
        self.df = float(_check_positive("df", df))
        if self.df <= self.do:
            raise ValidationError(f"fin diameter df ({self.df}) must be > tube od do ({self.do})")
        self.n_fins = float(_check_positive("n_fins_per_m", n_fins_per_m))
        self.tf = float(_check_positive("fin_thickness", fin_thickness))

        A_fin = np.pi / 4 * (self.df**2 - self.do**2) * 2 * self.n_fins * self.L * self.n_tubes
        A_tube = np.pi * self.do * (1 - self.n_fins * self.tf) * self.L * self.n_tubes
        if A_tube < 0:
            raise ValidationError("fin_thickness * n_fins_per_m exceeds available tube length fraction")
        self.A = A_fin + A_tube

    def overall_U(self, hi=8000, ho=50, eta_fin=0.85, Rf=0.0002):
        _check_positive("hi", hi)
        _check_positive("ho", ho)
        if not (0 < eta_fin <= 1):
            raise ValidationError(f"eta_fin must be in (0, 1], got {eta_fin}")
        Rf = max(float(Rf), 0.0)
        U = 1.0 / ((self.A / (np.pi * self.do * self.L * self.n_tubes)) * (1/hi) +
                   1/(eta_fin * ho) + Rf)
        if not np.isfinite(U) or U <= 0:
            raise ValidationError(f"Computed U is invalid: {U}")
        return float(U)

    def steady_state(self, m_h, cp_h, Th_in, m_c, cp_c, Tc_in, U=None):
        _check_positive("m_h", m_h)
        _check_positive("cp_h", cp_h)
        _check_positive("m_c", m_c)
        _check_positive("cp_c", cp_c)
        _check_temperature_pair(Th_in, Tc_in)
        if U is None:
            U = self.overall_U()
        else:
            _check_positive("U", U)
        C_h, C_c = m_h * cp_h, m_c * cp_c
        C_min = min(C_h, C_c)
        Cr = C_min / max(C_h, C_c)
        NTU = U * self.A / C_min
        eps = self.effectiveness_ntu(NTU, Cr, 'cross')
        Q = eps * C_min * (Th_in - Tc_in)
        return {
            'Q': Q, 'eps': eps, 'NTU': NTU,
            'Th_out': Th_in - Q / C_h,
            'Tc_out': Tc_in + Q / C_c, 'U': U, 'A': self.A
        }


print("Heat-exchanger classes defined with input validation: ShellTube | Plate | FinnedTube")
print("Invalid geometry, flows, temperatures or U now raise ValidationError with a clear message.")


## 3. Instantiation & Steady-State Comparison

### 3D Schematic — Shell-and-Tube Heat Exchanger

![3D Schematic — Shell-and-Tube Heat Exchanger](06_3d_shell_tube_schematic.png)

*Simplified 3-D view showing shell, tubes and baffles.*


In [ ]:
# Create units
hx_shell = ShellTubeHeatExchanger(n_tubes=100, tube_length=3.0, tube_od=0.025)
hx_plate = PlateHeatExchanger(n_plates=60, plate_area=0.25)
hx_fin   = FinnedTubeHeatExchanger(n_tubes=80, L=3.5)

# Common process conditions
m_h, cp_h, Th_in = 2.5, 4180, 120.0
m_c, cp_c, Tc_in = 3.0, 4180, 25.0

results = {}
for hx in [hx_shell, hx_plate, hx_fin]:
    results[hx.name] = hx.steady_state(m_h, cp_h, Th_in, m_c, cp_c, Tc_in)

df_ss = pd.DataFrame(results).T[['Q', 'eps', 'NTU', 'Th_out', 'Tc_out', 'U', 'A']]
df_ss['Q_kW'] = df_ss['Q'] / 1000
print("=== Steady-State Performance Comparison ===")
print(df_ss.round(3).to_string())

# Bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df_ss['Q_kW'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='k')
axes[0].set_ylabel('Heat Duty [kW]'); axes[0].set_title('Heat Duty')
df_ss['eps'].plot(kind='bar', ax=axes[1], color='seagreen', edgecolor='k')
axes[1].set_ylabel('Effectiveness ε'); axes[1].set_title('Effectiveness')
df_ss['U'].plot(kind='bar', ax=axes[2], color='coral', edgecolor='k')
axes[2].set_ylabel('U [W/m²K]'); axes[2].set_title('Overall U')
for ax in axes:
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# --- Input validation demos (safe to run) ---
print("=== INPUT VALIDATION DEMOS ===")

def try_call(desc, fn):
    try:
        fn()
        print(f"  UNEXPECTED PASS: {desc}")
    except ValidationError as e:
        print(f"  OK caught: {desc}")
        print(f"       → {e}")
    except Exception as e:
        print(f"  Other error ({type(e).__name__}): {desc} → {e}")

try_call("negative mass flow", lambda: hx_shell.steady_state(-1, 4180, 120, 3, 4180, 25))
try_call("cold hotter than hot", lambda: hx_shell.steady_state(2.5, 4180, 40, 3, 4180, 80))
try_call("tube_id >= tube_od", lambda: ShellTubeHeatExchanger(tube_od=0.02, tube_id=0.025))
try_call("zero area plates", lambda: PlateHeatExchanger(n_plates=0))
try_call("fin diameter < tube od", lambda: FinnedTubeHeatExchanger(do=0.03, df=0.02))

print("\nValid call still works:")
r = hx_shell.steady_state(2.5, 4180, 120, 3.0, 4180, 25, U=800)
print(f"  Q = {r['Q']/1000:.1f} kW, ε = {r['eps']:.3f}")


### Steady-State Performance Comparison

![Steady-State Performance Comparison](01_steady_state_comparison.png)

*Heat duty, effectiveness and overall U for Shell-Tube, Plate and Finned-Tube under identical process conditions.*


## 4. Transient Analysis (Step Response)

In [ ]:
t, Th_out, Th_in_profile, tau = hx_shell.transient_response(
    m_h=2.5, cp_h=4180, m_c=2.0, cp_c=4180, U=800,
    Th_in_step=(100, 140), Tc_in=20, t_step=60, t_final=600
)

print("=== TRANSIENT ANALYSIS — Step Response ===")
print(f"Time constant τ          : {tau:.1f} s")
print(f"Settling time (≈4τ)      : {4*tau:.1f} s")
print(f"Initial Th_out           : {Th_out[0]:.2f} °C")
print(f"Final Th_out             : {Th_out[-1]:.2f} °C")
print(f"Rise time (10–90 %)      : ~{2.2*tau:.0f} s")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t, Th_in_profile, 'r-', lw=2.5, label='Th_in (step input)', alpha=0.85)
ax.plot(t, Th_out, 'b-', lw=2.5, label='Th_out (response)')
ax.axhline(Th_out[0], color='b', ls=':', alpha=0.5, label='Initial SS')
ax.axhline(Th_out[-1], color='b', ls='--', alpha=0.5, label='Final SS')
ax.axvline(60, color='gray', ls='--', alpha=0.6)
ax.text(70, 115, 'Step at t = 60 s', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
ax.set_xlabel('Time [s]')
ax.set_ylabel('Temperature [°C]')
ax.set_title('Transient Response — Step Change in Hot Inlet Temperature')
ax.legend(loc='best')
plt.tight_layout()
plt.show()

print("\nConclusion: First-order lag behaviour. ~4τ needed to reach new steady state.")

### Transient Step Response

![Transient Step Response](02_transient_response.png)

*First-order lag response of hot-fluid outlet temperature to a step change in inlet temperature.*


## 5. ML Fouling Prediction (Kern–Seaton + Random Forest)

In [ ]:
print("=== ML FOULING PREDICTION MODEL ===")
print("Generating synthetic operational data (Kern–Seaton) ...\n")

fouling_data = hx_shell.generate_fouling_data(n_samples=2500)
model, r2, mae, X_test, y_test, y_pred = hx_shell.train_fouling_ml_model(fouling_data)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Model Performance (held-out test set)")
print(f"  R²   : {r2:.4f}")
print(f"  MAE  : {mae:.6f} m²K/W")
print(f"  RMSE : {rmse:.6f} m²K/W")

# Feature importance
feat_names = ['Hours', 'm_hot', 'T_hot_in', 'Hardness', 'pH', 'Velocity']
imp = model.feature_importances_
feat_imp = pd.DataFrame({'Feature': feat_names, 'Importance': imp})\
            .sort_values('Importance', ascending=False)
print("\nFeature Importance:")
print(feat_imp.to_string(index=False))

# Example prediction
test_case = pd.DataFrame({
    'operating_hours': [20000],
    'm_dot_hot': [2.5],
    'T_hot_in': [150],
    'water_hardness': [300],
    'pH': [7.5],
    'velocity': [1.5]
})
Rf_pred = model.predict(test_case)[0]
U_clean = 1000.0
U_fouled = 1 / (1/U_clean + Rf_pred)
deg = (U_clean - U_fouled) / U_clean * 100

print("\n=== PREDICTION EXAMPLE ===")
print(f"Input  : 20 000 h, 300 ppm hardness, 1.5 m/s")
print(f"Rf     : {Rf_pred:.6f} m²K/W")
print(f"U_clean: {U_clean:.0f} → U_fouled: {U_fouled:.1f} W/m²K")
print(f"Degradation: {deg:.1f} %")

# Plots
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(y_test, y_pred, alpha=0.45, s=18, c='royalblue', edgecolors='k', lw=0.3)
lims = [0, max(y_test.max(), y_pred.max()) * 1.05]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect')
axes[0].set_xlabel('Actual Rf [m²K/W]'); axes[0].set_ylabel('Predicted Rf')
axes[0].set_title(f'Parity Plot  (R² = {r2:.3f})'); axes[0].legend()

axes[1].barh(feat_imp['Feature'], feat_imp['Importance'], color='coral', edgecolor='k')
axes[1].set_xlabel('Importance'); axes[1].set_title('Feature Importance')

hours = np.linspace(0, 40000, 120)
for h in [100, 300, 500]:
    df_curve = pd.DataFrame({
        'operating_hours': hours, 'm_dot_hot': 2.5, 'T_hot_in': 150,
        'water_hardness': h, 'pH': 7.5, 'velocity': 1.5
    })
    axes[2].plot(hours, model.predict(df_curve), lw=2, label=f'{h} ppm')
axes[2].set_xlabel('Operating Hours'); axes[2].set_ylabel('Rf [m²K/W]')
axes[2].set_title('Fouling Growth vs Hardness'); axes[2].legend()

plt.tight_layout()
plt.show()

print("\nConclusion: Hardness & operating time dominate fouling → water treatment is critical.")

### ML Fouling Prediction (Random Forest)

![ML Fouling Prediction (Random Forest)](03_fouling_ml_prediction.png)

*Parity plot, feature importance and fouling growth curves vs water hardness.*


## 6. Uncertainty Quantification (Monte Carlo)

In [ ]:
print("=== MONTE-CARLO UNCERTAINTY PROPAGATION ===")

n_mc = 3000
# Parameter distributions (relative std-dev)
U_samples   = np.random.normal(800, 80, n_mc)          # ±10 %
A_samples   = np.random.normal(hx_shell.A, hx_shell.A*0.03, n_mc)
m_h_samples = np.random.normal(2.5, 0.1, n_mc)
m_c_samples = np.random.normal(3.0, 0.12, n_mc)
Th_samples  = np.random.normal(120, 2, n_mc)
Tc_samples  = np.random.normal(25, 1.5, n_mc)

Q_mc, eps_mc = [], []
for i in range(n_mc):
    # temporarily override area
    orig_A = hx_shell.A
    hx_shell.A = max(A_samples[i], 1.0)
    r = hx_shell.steady_state(m_h_samples[i], 4180, Th_samples[i],
                             m_c_samples[i], 4180, Tc_samples[i],
                             U=max(U_samples[i], 100))
    hx_shell.A = orig_A
    Q_mc.append(r['Q'] / 1000)   # kW
    eps_mc.append(r['eps'])

Q_mc = np.array(Q_mc)
eps_mc = np.array(eps_mc)

print(f"Heat duty  Q : {Q_mc.mean():.2f} ± {Q_mc.std():.2f} kW  "
      f"(95 % CI [{np.percentile(Q_mc,2.5):.2f}, {np.percentile(Q_mc,97.5):.2f}])")
print(f"Effectiveness: {eps_mc.mean():.3f} ± {eps_mc.std():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(Q_mc, bins=40, color='steelblue', edgecolor='k', alpha=0.8)
axes[0].axvline(Q_mc.mean(), color='r', ls='--', label=f'Mean = {Q_mc.mean():.1f} kW')
axes[0].set_xlabel('Q [kW]'); axes[0].set_title('Heat Duty Distribution'); axes[0].legend()

axes[1].hist(eps_mc, bins=40, color='seagreen', edgecolor='k', alpha=0.8)
axes[1].axvline(eps_mc.mean(), color='r', ls='--', label=f'Mean = {eps_mc.mean():.3f}')
axes[1].set_xlabel('ε'); axes[1].set_title('Effectiveness Distribution'); axes[1].legend()
plt.tight_layout()
plt.show()

### Monte-Carlo Uncertainty Propagation

![Monte-Carlo Uncertainty Propagation](04_monte_carlo_uncertainty.png)

*Distributions of heat duty and effectiveness under parameter uncertainty.*


## 7. Sensitivity Analysis

In [ ]:
print("=== LOCAL SENSITIVITY (One-at-a-time ±10 %) ===")

base = hx_shell.steady_state(2.5, 4180, 120, 3.0, 4180, 25, U=800)
base_Q = base['Q']

params = {
    'U': 800,
    'm_h': 2.5,
    'm_c': 3.0,
    'Th_in': 120,
    'Tc_in': 25,
    'n_tubes': 100
}

sens = {}
for p, val in params.items():
    # +10 %
    kwargs = dict(m_h=2.5, cp_h=4180, Th_in=120, m_c=3.0, cp_c=4180, Tc_in=25, U=800)
    if p == 'n_tubes':
        hx_tmp = ShellTubeHeatExchanger(n_tubes=int(val*1.1), tube_length=3.0)
        Q_hi = hx_tmp.steady_state(**{k: v for k, v in kwargs.items() if k != 'U'}, U=kwargs['U'])['Q']
        hx_tmp = ShellTubeHeatExchanger(n_tubes=int(val*0.9), tube_length=3.0)
        Q_lo = hx_tmp.steady_state(**{k: v for k, v in kwargs.items() if k != 'U'}, U=kwargs['U'])['Q']
    else:
        kwargs[p] = val * 1.1
        Q_hi = hx_shell.steady_state(**kwargs)['Q']
        kwargs[p] = val * 0.9
        Q_lo = hx_shell.steady_state(**kwargs)['Q']
    sens[p] = ((Q_hi - Q_lo) / 2) / base_Q * 100   # % change in Q per 10 % param change

sens_df = pd.Series(sens).sort_values(key=abs, ascending=False)
print(sens_df.round(2).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['crimson' if v < 0 else 'seagreen' for v in sens_df]
sens_df.plot(kind='barh', ax=ax, color=colors, edgecolor='k')
ax.set_xlabel('ΔQ / Q  [%] for ±10 % parameter change')
ax.set_title('Local Sensitivity Ranking')
plt.tight_layout()
plt.show()

### Local Sensitivity Ranking

![Local Sensitivity Ranking](05_sensitivity_analysis.png)

*Percentage change in heat duty for a ±10 % variation of each parameter.*


## 8. Design Optimization

In [ ]:
print("=== OPTIMIZATION: Maximize heat duty subject to area & velocity constraints ===")

def objective(x):
    """x = [n_tubes, tube_length, tube_od]"""
    n, L, do = x
    hx = ShellTubeHeatExchanger(n_tubes=int(n), tube_length=L, tube_od=do, tube_id=do*0.8)
    # Approximate tube-side velocity constraint (keep < 2.5 m/s)
    area_flow = n * np.pi * (do*0.8/2)**2
    vel = 2.5 / (1000 * area_flow)          # rough volumetric flow / area
    if vel > 2.5:
        return 1e6                          # penalty
    res = hx.steady_state(2.5, 4180, 120, 3.0, 4180, 25, U=850)
    return -res['Q']                        # maximize Q → minimize -Q

bounds = [(40, 200), (1.5, 6.0), (0.015, 0.035)]
result = differential_evolution(objective, bounds, seed=42, popsize=15, mutation=0.7)

n_opt, L_opt, do_opt = result.x
hx_opt = ShellTubeHeatExchanger(n_tubes=int(n_opt), tube_length=L_opt, tube_od=do_opt)
res_opt = hx_opt.steady_state(2.5, 4180, 120, 3.0, 4180, 25, U=850)

print(f"Optimal design")
print(f"  n_tubes     : {int(n_opt)}")
print(f"  Length      : {L_opt:.2f} m")
print(f"  Tube OD     : {do_opt*1000:.1f} mm")
print(f"  Area        : {hx_opt.A:.2f} m²")
print(f"  Heat duty   : {res_opt['Q']/1000:.1f} kW")
print(f"  Effectiveness: {res_opt['eps']:.3f}")

## 9. Summary Dashboard

In [ ]:
print("=" * 60)
print("HEAT EXCHANGER ANALYSIS — EXECUTIVE SUMMARY")
print("=" * 60)
print(f"\n1. Steady-state comparison (identical process conditions)")
print(df_ss[['Q_kW', 'eps', 'U', 'A']].round(2).to_string())
print(f"\n2. Transient: τ ≈ {tau:.0f} s → settling ~{4*tau:.0f} s")
print(f"3. Fouling ML model: R² = {r2:.3f}, key drivers = Hardness + Time")
print(f"4. Uncertainty (Q): {Q_mc.mean():.1f} ± {Q_mc.std():.1f} kW (1σ)")
print(f"5. Most sensitive parameter: {sens_df.index[0]} (ΔQ/Q ≈ {sens_df.iloc[0]:.1f} %)")
print(f"6. Optimized shell-tube: {int(n_opt)} tubes × {L_opt:.1f} m → {res_opt['Q']/1000:.0f} kW")
print("\nUpgrade complete. Notebook is fully self-contained and ready for further extension")
print("(real plant data ingestion, CFD boundary conditions, digital-twin coupling, etc.).")

## 10. Predictive Maintenance Suite — Fouling & Scaling Early Detection

This module adds industrial-grade monitoring and prognostics:

| Feature | Description |
|---------|-------------|
| Early detection | Anomaly scoring on Rf, ΔP and efficiency |
| Dual ML models | Random Forest + XGBoost |
| Maintenance warnings | 7-day and 30-day horizon alerts |
| Efficiency tracking | Continuous heat-transfer degradation |
| Pressure-drop monitoring | ΔP rise linked to fouling layer |
| RUL estimation | Remaining Useful Life until critical Rf or ΔP |
| Dashboard | Single-view status for operators |


In [ ]:
# Extra imports for predictive maintenance
from sklearn.ensemble import GradientBoostingRegressor, IsolationForest
from sklearn.preprocessing import StandardScaler
from datetime import datetime, timedelta
import xgboost as xgb

print("Predictive-maintenance libraries ready (RF + XGBoost + IsolationForest).")


### 10.1 Synthetic sensor time-series (plant-like data)

We simulate 180 days of daily sensor readings that would come from a real shell-and-tube HX:
operating hours, flow, temperatures, estimated Rf, overall U, heat duty, pressure drop, and efficiency.


In [ ]:
print("=== GENERATING 180-DAY SENSOR TIME-SERIES ===")

np.random.seed(42)
n_days = 180
dates = pd.date_range("2026-01-01", periods=n_days, freq="D")

# Progressive fouling (Kern–Seaton style) + seasonal / operational noise
hours = np.arange(n_days) * 24.0
hardness = 280 + 40 * np.sin(2 * np.pi * np.arange(n_days) / 90)   # mild seasonal variation
velocity = 1.4 + 0.15 * np.random.randn(n_days)
velocity = np.clip(velocity, 0.8, 2.2)
T_hot = 145 + 8 * np.sin(2 * np.pi * np.arange(n_days) / 60) + np.random.randn(n_days)

Rf_inf = 1.15e-4 * (hardness / 200) ** 1.1 * np.exp(0.011 * (T_hot - 100))
tau_f = 8500 * (velocity / 1.5) ** 0.75 / (hardness / 200) ** 0.28
Rf_true = Rf_inf * (1 - np.exp(-hours / tau_f))
Rf_true += np.random.normal(0, 3e-6, n_days)
Rf_true = np.clip(Rf_true, 0, None)

U_clean = 950.0
U = 1.0 / (1.0 / U_clean + Rf_true)
Q_clean = 680.0   # kW reference
Q = Q_clean * (U / U_clean) * (0.98 + 0.02 * np.random.randn(n_days))
efficiency = Q / Q_clean * 100

# Pressure drop rises with fouling layer (simple linear + quadratic term)
dP0 = 28.0   # kPa clean
dP = dP0 * (1 + 180 * Rf_true + 4.5e5 * Rf_true ** 2) + np.random.normal(0, 0.6, n_days)

sensor_df = pd.DataFrame({
    "date": dates,
    "operating_hours": hours,
    "water_hardness": hardness,
    "velocity": velocity,
    "T_hot_in": T_hot,
    "Rf": Rf_true,
    "U": U,
    "Q_kW": Q,
    "efficiency_pct": efficiency,
    "delta_P_kPa": dP
})

print(sensor_df[["date", "Rf", "U", "efficiency_pct", "delta_P_kPa"]].head(3).to_string(index=False))
print("...")
print(sensor_df[["date", "Rf", "U", "efficiency_pct", "delta_P_kPa"]].tail(3).to_string(index=False))
print(f"\nGenerated {len(sensor_df)} daily records.")


### 10.2 Dual-model prediction (Random Forest + XGBoost)

Both models predict next-day fouling resistance \(R_f\) from process variables.
We also train models that forecast \(R_f\) 7 and 30 days ahead for maintenance planning.


In [ ]:
print("=== TRAINING RF + XGBoost MODELS ===")

# Lag features improve short-term forecasts on trending fouling data
sensor_df["Rf_lag1"] = sensor_df["Rf"].shift(1)
sensor_df["Rf_lag3"] = sensor_df["Rf"].shift(3)
sensor_df["dRf"] = sensor_df["Rf"].diff()
sensor_df = sensor_df.dropna().reset_index(drop=True)

features = ["operating_hours", "water_hardness", "velocity", "T_hot_in", "Rf_lag1", "Rf_lag3", "dRf"]
target = "Rf"

X = sensor_df[features]
y = sensor_df[target]
# Time-series split (no shuffle)
split = int(len(X) * 0.75)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

rf = RandomForestRegressor(n_estimators=200, max_depth=10, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

xgb_model = xgb.XGBRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.08,
    subsample=0.85, colsample_bytree=0.85, random_state=42, n_jobs=-1, verbosity=0
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

print("1-day-ahead performance (held-out last 25 %):")
print(f"  Random Forest  R² = {r2_score(y_test, rf_pred):.4f}  |  MAE = {mean_absolute_error(y_test, rf_pred):.2e}")
print(f"  XGBoost        R² = {r2_score(y_test, xgb_pred):.4f}  |  MAE = {mean_absolute_error(y_test, xgb_pred):.2e}")

# --- Multi-horizon targets (7-day & 30-day ahead) ---
sensor_df["Rf_7d"]  = sensor_df["Rf"].shift(-7)
sensor_df["Rf_30d"] = sensor_df["Rf"].shift(-30)
df_h = sensor_df.dropna(subset=["Rf_7d", "Rf_30d"]).copy()

X_h = df_h[features]
y7  = df_h["Rf_7d"]
y30 = df_h["Rf_30d"]

rf7  = RandomForestRegressor(n_estimators=150, max_depth=8, random_state=42, n_jobs=-1).fit(X_h, y7)
rf30 = RandomForestRegressor(n_estimators=150, max_depth=8, random_state=42, n_jobs=-1).fit(X_h, y30)
xgb7  = xgb.XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0).fit(X_h, y7)
xgb30 = xgb.XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0).fit(X_h, y30)

print("\nMulti-horizon models trained (RF + XGBoost for 7-day and 30-day horizons).")


### 10.3 Early detection, 7-day / 30-day warnings & RUL


In [ ]:
print("=== EARLY DETECTION + MAINTENANCE WARNINGS + RUL ===")

# Critical thresholds (industrial rules of thumb)
Rf_CRIT   = 1.8e-4      # m²K/W  → schedule cleaning
dP_CRIT   = 55.0        # kPa
EFF_CRIT  = 78.0        # % of clean duty

# Current (last day) state
last = sensor_df.iloc[-1]
print(f"Latest sensor snapshot ({last['date'].date()}):")
print(f"  Rf          = {last['Rf']:.6f} m²K/W")
print(f"  ΔP          = {last['delta_P_kPa']:.1f} kPa")
print(f"  Efficiency  = {last['efficiency_pct']:.1f} %")
print(f"  U           = {last['U']:.1f} W/m²K")

# Ensemble prediction (average of RF & XGBoost)
# Rebuild current feature vector with lags
X_now = pd.DataFrame([{
    "operating_hours": last["operating_hours"],
    "water_hardness": last["water_hardness"],
    "velocity": last["velocity"],
    "T_hot_in": last["T_hot_in"],
    "Rf_lag1": sensor_df["Rf"].iloc[-2] if len(sensor_df) > 1 else last["Rf"],
    "Rf_lag3": sensor_df["Rf"].iloc[-4] if len(sensor_df) > 3 else last["Rf"],
    "dRf": sensor_df["Rf"].iloc[-1] - sensor_df["Rf"].iloc[-2] if len(sensor_df) > 1 else 0.0
}])[features]
Rf_now_pred = 0.5 * (rf.predict(X_now)[0] + xgb_model.predict(X_now)[0])
Rf_7d_pred  = 0.5 * (rf7.predict(X_now)[0]  + xgb7.predict(X_now)[0])
Rf_30d_pred = 0.5 * (rf30.predict(X_now)[0] + xgb30.predict(X_now)[0])

print(f"\nPredicted Rf:")
print(f"  Next day   : {Rf_now_pred:.6f}")
print(f"  +7 days    : {Rf_7d_pred:.6f}")
print(f"  +30 days   : {Rf_30d_pred:.6f}")

# Warning logic
def warning_level(rf_val, dp_val, eff_val):
    score = 0
    msgs = []
    if rf_val > Rf_CRIT * 0.70:
        score += 1; msgs.append("Rf elevated")
    if rf_val > Rf_CRIT * 0.90:
        score += 1; msgs.append("Rf near critical")
    if dp_val > dP_CRIT * 0.75:
        score += 1; msgs.append("ΔP rising")
    if eff_val < EFF_CRIT + 5:
        score += 1; msgs.append("Efficiency declining")
    if score == 0:
        return "GREEN", "Normal operation"
    if score <= 2:
        return "YELLOW", "; ".join(msgs)
    return "RED", "; ".join(msgs)

status_now, msg_now = warning_level(last["Rf"], last["delta_P_kPa"], last["efficiency_pct"])
status_7d,  _ = warning_level(Rf_7d_pred, last["delta_P_kPa"] * 1.05, last["efficiency_pct"] * 0.97)
status_30d, _ = warning_level(Rf_30d_pred, last["delta_P_kPa"] * 1.15, last["efficiency_pct"] * 0.93)

print(f"\n--- Maintenance Status ---")
print(f"  Current   : {status_now:6s}  → {msg_now}")
print(f"  7-day     : {status_7d:6s}")
print(f"  30-day    : {status_30d:6s}")

# Simple RUL estimation (linear extrapolation of Rf to critical)
# Fit recent trend
recent = sensor_df.tail(45)
slope = np.polyfit(recent["operating_hours"], recent["Rf"], 1)[0]
if slope > 1e-10:
    hours_to_crit = max(0, (Rf_CRIT - last["Rf"]) / slope)
    days_to_crit  = hours_to_crit / 24.0
else:
    days_to_crit = 999.0

print(f"\nRemaining Useful Life (RUL) estimate")
print(f"  Days until Rf reaches critical ({Rf_CRIT:.1e}): ≈ {days_to_crit:.0f} days")
if days_to_crit < 7:
    print("  >>> IMMEDIATE maintenance recommended")
elif days_to_crit < 30:
    print("  >>> Plan cleaning within the next maintenance window")
else:
    print("  >>> No urgent action required")


### 10.4 Anomaly detection (Isolation Forest) for early fouling / scaling


In [ ]:
print("=== ANOMALY DETECTION (Isolation Forest) ===")

# Features that indicate abnormal fouling behaviour
anom_features = ["Rf", "delta_P_kPa", "efficiency_pct", "U"]
scaler = StandardScaler()
X_anom = scaler.fit_transform(sensor_df[anom_features])

iso = IsolationForest(n_estimators=200, contamination=0.06, random_state=42)
sensor_df["anomaly"] = iso.fit_predict(X_anom)          # -1 = anomaly
sensor_df["anomaly_score"] = iso.decision_function(X_anom)

n_anom = (sensor_df["anomaly"] == -1).sum()
print(f"Detected {n_anom} anomalous days ({100*n_anom/len(sensor_df):.1f} % of record).")

# Show most recent anomalies
recent_anom = sensor_df[sensor_df["anomaly"] == -1].tail(5)
if len(recent_anom):
    print("\nMost recent anomalies:")
    print(recent_anom[["date", "Rf", "delta_P_kPa", "efficiency_pct", "anomaly_score"]].to_string(index=False))
else:
    print("No recent anomalies.")


### 10.5 Degradation & pressure-drop tracking plots


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# 1. Rf & efficiency over time
ax = axes[0, 0]
ax.plot(sensor_df["date"], sensor_df["Rf"] * 1e4, "b-", lw=1.8, label="Rf × 10⁴")
ax.axhline(Rf_CRIT * 1e4, color="r", ls="--", label="Critical Rf")
ax.set_ylabel("Rf × 10⁴  [m²K/W]")
ax.set_title("Fouling Resistance Growth")
ax.legend(loc="upper left")
ax2 = ax.twinx()
ax2.plot(sensor_df["date"], sensor_df["efficiency_pct"], "g-", alpha=0.7, label="Efficiency %")
ax2.set_ylabel("Efficiency [%]")
ax2.legend(loc="upper right")

# 2. Pressure drop
ax = axes[0, 1]
ax.plot(sensor_df["date"], sensor_df["delta_P_kPa"], "m-", lw=1.8)
ax.axhline(dP_CRIT, color="r", ls="--", label="Critical ΔP")
ax.set_ylabel("ΔP [kPa]")
ax.set_title("Pressure-Drop Monitoring")
ax.legend()

# 3. U degradation
ax = axes[1, 0]
ax.plot(sensor_df["date"], sensor_df["U"], "c-", lw=1.8)
ax.axhline(U_clean * 0.75, color="orange", ls="--", label="75 % of clean U")
ax.set_ylabel("U [W/m²K]")
ax.set_title("Overall Heat-Transfer Coefficient")
ax.legend()

# 4. Anomaly scores
ax = axes[1, 1]
colors = np.where(sensor_df["anomaly"] == -1, "red", "steelblue")
ax.scatter(sensor_df["date"], sensor_df["anomaly_score"], c=colors, s=12, alpha=0.7)
ax.axhline(0, color="k", ls=":", alpha=0.5)
ax.set_ylabel("Anomaly score (lower = more anomalous)")
ax.set_title("Isolation-Forest Early Detection")
ax.set_xlabel("Date")

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()
print("Degradation tracking plots generated.")


### Degradation & Pressure-Drop Tracking

![Degradation & Pressure-Drop Tracking](07_degradation_tracking.png)

*180-day evolution of Rf, efficiency, ΔP and overall U.*


### 10.6 Predictive Maintenance Dashboard


In [ ]:
print("=" * 72)
print("          PREDICTIVE MAINTENANCE DASHBOARD — SHELL-TUBE HX")
print("=" * 72)

print(f"\nReport generated : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"Data coverage    : {sensor_df['date'].iloc[0].date()} → {sensor_df['date'].iloc[-1].date()} ({n_days} days)")

print("\n┌─ CURRENT HEALTH ─────────────────────────────────────┐")
print(f"│  Status            : {status_now}")
print(f"│  Fouling Rf        : {last['Rf']:.6f} m²K/W")
print(f"│  Pressure drop ΔP  : {last['delta_P_kPa']:.1f} kPa")
print(f"│  Efficiency        : {last['efficiency_pct']:.1f} % of clean")
print(f"│  Overall U         : {last['U']:.1f} W/m²K")
print("└──────────────────────────────────────────────────────┘")

print("\n┌─ FORECAST ───────────────────────────────────────────┐")
print(f"│  Rf in 7 days      : {Rf_7d_pred:.6f}   → status {status_7d}")
print(f"│  Rf in 30 days     : {Rf_30d_pred:.6f}  → status {status_30d}")
print(f"│  Estimated RUL     : ≈ {days_to_crit:.0f} days until critical Rf")
print("└──────────────────────────────────────────────────────┘")

print("\n┌─ MODEL PERFORMANCE (1-day ahead) ────────────────────┐")
print(f"│  Random Forest R²  : {r2_score(y_test, rf_pred):.4f}")
print(f"│  XGBoost R²        : {r2_score(y_test, xgb_pred):.4f}")
print("└──────────────────────────────────────────────────────┘")

print("\n┌─ RECOMMENDED ACTIONS ────────────────────────────────┐")
if days_to_crit < 7 or status_now == "RED":
    print("│  ★ Schedule chemical / mechanical cleaning ASAP")
    print("│  ★ Reduce load if possible until cleaned")
    print("│  ★ Inspect for scaling / biofouling samples")
elif days_to_crit < 30 or status_7d in ("YELLOW", "RED"):
    print("│  ★ Plan cleaning in next 1–2 weeks")
    print("│  ★ Increase monitoring frequency (daily ΔP & U)")
    print("│  ★ Check water treatment (hardness / pH)")
else:
    print("│  ★ Continue normal operation")
    print("│  ★ Maintain water-treatment program")
    print("│  ★ Next full review in 30 days")
print("└──────────────────────────────────────────────────────┘")

print("\nDashboard complete. All predictive-maintenance features active.")


## 11. Nanofluid Cooling Impact on HX Performance

Compare base fluid (water) vs Al2O3, CuO and hybrid nanofluids using Maxwell conductivity and mixture rules. Quantify effect on overall U, heat duty and required pumping power.


In [ ]:
print("=== NANOFLUID PROPERTY & PERFORMANCE MODULE ===")

props = {
    "water":   {"k": 0.613, "mu": 8.9e-4, "rho": 997, "cp": 4180},
    "Al2O3":   {"k": 40.0,  "rho": 3970, "cp": 765},
    "CuO":     {"k": 33.0,  "rho": 6315, "cp": 535},
    "TiO2":    {"k": 8.4,   "rho": 4250, "cp": 686},
}

def maxwell_k(k_bf, k_p, phi):
    return k_bf * (k_p + 2*k_bf + 2*phi*(k_p - k_bf)) / (k_p + 2*k_bf - phi*(k_p - k_bf))

def mixture_rho(rho_bf, rho_p, phi):
    return (1 - phi)*rho_bf + phi*rho_p

def mixture_cp(rho_bf, cp_bf, rho_p, cp_p, phi, rho_nf):
    return ((1 - phi)*rho_bf*cp_bf + phi*rho_p*cp_p) / rho_nf

def brinkman_mu(mu_bf, phi):
    return mu_bf / (1 - phi)**2.5

def nanofluid_row(name, k_p, rho_p, cp_p, phi, base="water"):
    bf = props[base]
    k_nf  = maxwell_k(bf["k"], k_p, phi)
    rho_nf = mixture_rho(bf["rho"], rho_p, phi)
    cp_nf  = mixture_cp(bf["rho"], bf["cp"], rho_p, cp_p, phi, rho_nf)
    mu_nf  = brinkman_mu(bf["mu"], phi)
    h_ratio = (k_nf/bf["k"])**0.6 * ((rho_nf*cp_nf)/(bf["rho"]*bf["cp"]))**0.4 * (bf["mu"]/mu_nf)**0.4
    return {
        "fluid": name, "phi_%": phi*100,
        "k_nf": k_nf, "mu_ratio": mu_nf/bf["mu"],
        "h_ratio": h_ratio, "rho": rho_nf, "cp": cp_nf
    }

rows = [nanofluid_row("Water (base)", 0.613, 997, 4180, 0.0)]
for phi in [0.005, 0.01, 0.02]:
    rows.append(nanofluid_row(f"Al2O3 {phi*100:.1f}%", props["Al2O3"]["k"], props["Al2O3"]["rho"], props["Al2O3"]["cp"], phi))
    rows.append(nanofluid_row(f"CuO {phi*100:.1f}%",   props["CuO"]["k"],   props["CuO"]["rho"],   props["CuO"]["cp"],   phi))
    k_h = 0.5*(props["Al2O3"]["k"]+props["CuO"]["k"])
    rho_h = 0.5*(props["Al2O3"]["rho"]+props["CuO"]["rho"])
    cp_h = 0.5*(props["Al2O3"]["cp"]+props["CuO"]["cp"])
    rows.append(nanofluid_row(f"Hybrid Al2O3-CuO {phi*100:.1f}%", k_h, rho_h, cp_h, phi))

nf_df = pd.DataFrame(rows)
print(nf_df.round(4).to_string(index=False))

base_hi = 6000.0
print("\nEffect on Shell-Tube overall U (ho fixed = 3000 W/m2K):")
rf_ref = 5e-5
for _, r in nf_df.iterrows():
    hi = base_hi * r["h_ratio"]
    U_nf = hx_shell.overall_U(hi=hi, ho=3000, Rf_i=0, Rf_o=rf_ref)
    print(f"  {r['fluid']:28s}  U approx {U_nf:.1f} W/m2K   (h_ratio={r['h_ratio']:.3f})")


## 12. Economic Analysis — Cleaning Cost vs Efficiency Loss

Estimate daily lost revenue from fouling and compare against cleaning cost to find optimal intervention point.


In [ ]:
print("=== ECONOMIC / COST-BENEFIT MODULE ===")

ENERGY_PRICE   = 0.12
CLEAN_COST     = 4500
CLEAN_DOWNTIME = 18
PRODUCTION_VALUE = 85
Q_CLEAN_KW     = 680.0

sensor_df["lost_Q_kW"] = Q_CLEAN_KW * (1 - sensor_df["efficiency_pct"]/100)
sensor_df["daily_loss_$"] = sensor_df["lost_Q_kW"] * 24 * ENERGY_PRICE

cum_loss = sensor_df["daily_loss_$"].cumsum()
print(f"Cumulative energy-loss cost over {n_days} days: ${cum_loss.iloc[-1]:,.0f}")

cross = sensor_df.loc[cum_loss > CLEAN_COST, "date"]
if len(cross):
    print(f"Cumulative loss exceeds one cleaning cost on: {cross.iloc[0].date()}")
else:
    print("Cumulative loss still below one cleaning cost in this window.")

intervals = [15, 30, 45, 60, 90, 120]
print("\nEstimated annual cost vs cleaning interval:")
print(f"{'Interval (d)':>14}  {'Cleanings/yr':>12}  {'Energy loss $/yr':>16}  {'Clean+DT $/yr':>14}  {'Total $/yr':>12}")
cost_list = []
for iv in intervals:
    n_clean = 365 / iv
    avg_loss_frac = 0.5 * min(0.22, iv / 200)
    energy = avg_loss_frac * Q_CLEAN_KW * 24 * 365 * ENERGY_PRICE
    clean_tot = n_clean * (CLEAN_COST + CLEAN_DOWNTIME * PRODUCTION_VALUE)
    total = energy + clean_tot
    cost_list.append(total)
    print(f"{iv:>14}  {n_clean:>12.1f}  {energy:>16,.0f}  {clean_tot:>14,.0f}  {total:>12,.0f}")

best_iv = intervals[int(np.argmin(cost_list))]
print(f"\nSuggested cleaning interval (under these assumptions): approx {best_iv} days")


## 13. What-If Scenario Simulator

Change hardness, velocity or inlet temperature and instantly see predicted Rf, efficiency and impact.


In [ ]:
print("=== WHAT-IF SCENARIO SIMULATOR ===")

def run_scenario(label, hardness=300, velocity=1.5, T_hot=150, hours_ahead=30*24):
    try:
        X_sc = pd.DataFrame([{
            "operating_hours": last["operating_hours"] + hours_ahead,
            "water_hardness": hardness,
            "velocity": velocity,
            "T_hot_in": T_hot,
            "Rf_lag1": last["Rf"],
            "Rf_lag3": last["Rf"],
            "dRf": 0.0
        }])[features]
        Rf_sc = 0.5 * (rf30.predict(X_sc)[0] + xgb30.predict(X_sc)[0])
    except Exception:
        Rf_inf = 1.15e-4 * (hardness/200)**1.1 * np.exp(0.011*(T_hot-100))
        tau_f = 8500 * (velocity/1.5)**0.75 / (hardness/200)**0.28
        t = last["operating_hours"] + hours_ahead
        Rf_sc = Rf_inf * (1 - np.exp(-t / tau_f))
    U_sc = 1 / (1/U_clean + Rf_sc)
    eff_sc = U_sc / U_clean * 100
    return {"scenario": label, "Rf": Rf_sc, "U": U_sc, "efficiency_%": eff_sc}

scenarios = [
    run_scenario("Baseline (current trend)", 300, 1.5, 150),
    run_scenario("Hardness +50 pct", 450, 1.5, 150),
    run_scenario("Hardness -30 pct (better treatment)", 210, 1.5, 150),
    run_scenario("Velocity +20 pct", 300, 1.8, 150),
    run_scenario("Lower hot-inlet T (-15 C)", 300, 1.5, 135),
    run_scenario("Aggressive (hard + hot)", 450, 1.2, 165),
]

sc_df = pd.DataFrame(scenarios)
print(sc_df.round(6).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
colors = ["steelblue", "crimson", "seagreen", "orange", "purple", "brown"]
ax.barh(sc_df["scenario"], sc_df["efficiency_%"], color=colors, edgecolor="k")
ax.axvline(last["efficiency_pct"], color="k", ls="--", label="Current efficiency")
ax.set_xlabel("Predicted efficiency in 30 days [%]")
ax.set_title("What-If: 30-day efficiency under different operating scenarios")
ax.legend()
plt.tight_layout()
plt.show()


## 14. Composite Equipment Health Index (0-100)

Combines Rf, delta-P, efficiency and anomaly score into a single operator-friendly health index.


In [ ]:
print("=== COMPOSITE HEALTH INDEX ===")

Rf_score  = np.clip(1 - sensor_df["Rf"] / Rf_CRIT, 0, 1)
dP_score  = np.clip(1 - sensor_df["delta_P_kPa"] / dP_CRIT, 0, 1)
eff_score = np.clip((sensor_df["efficiency_pct"] - 70) / 30, 0, 1)
if "anomaly_score" in sensor_df.columns:
    anom_score = np.clip((sensor_df["anomaly_score"] + 0.3) / 0.6, 0, 1)
else:
    anom_score = 0.8

sensor_df["health_index"] = (0.35*Rf_score + 0.25*dP_score + 0.25*eff_score + 0.15*anom_score) * 100

print(f"Current Health Index: {sensor_df['health_index'].iloc[-1]:.1f} / 100")
print(f"30-day average HI   : {sensor_df['health_index'].tail(30).mean():.1f}")
print(f"Minimum HI in record: {sensor_df['health_index'].min():.1f}")

fig, ax = plt.subplots(figsize=(12, 4))
ax.fill_between(sensor_df["date"], sensor_df["health_index"], alpha=0.4, color="steelblue")
ax.plot(sensor_df["date"], sensor_df["health_index"], color="navy", lw=1.5)
ax.axhline(80, color="g", ls="--", label="Healthy >= 80")
ax.axhline(60, color="orange", ls="--", label="Watch < 60")
ax.axhline(40, color="r", ls="--", label="Action < 40")
ax.set_ylabel("Health Index")
ax.set_title("Equipment Health Index over time")
ax.legend(loc="lower left")
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()


### Composite Equipment Health Index (0–100)

![Composite Equipment Health Index (0–100)](08_health_index.png)

*Health index combining fouling, pressure drop, efficiency and anomaly score.*


## 15. Model Explainability — Permutation Importance

Which process variables most influence the fouling prediction?


In [ ]:
print("=== PERMUTATION FEATURE IMPORTANCE (Random Forest) ===")

from sklearn.inspection import permutation_importance

r = permutation_importance(rf, X_test, y_test, n_repeats=12, random_state=42, n_jobs=-1)
imp_df = pd.DataFrame({
    "feature": features,
    "importance": r.importances_mean,
    "std": r.importances_std
}).sort_values("importance", ascending=True)

print(imp_df.round(5).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp_df["feature"], imp_df["importance"], xerr=imp_df["std"], color="coral", edgecolor="k")
ax.set_xlabel("Permutation importance")
ax.set_title("What drives Rf predictions? (higher = more influential)")
plt.tight_layout()
plt.show()


## 16. Integrated Operator Dashboard (All Features)


In [ ]:
print("=" * 78)
print("     HEAT EXCHANGER — INTEGRATED PREDICTIVE MAINTENANCE & PERFORMANCE DASHBOARD")
print("=" * 78)
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}   |   Asset: Shell-Tube HX")
print("-" * 78)

hi_now = sensor_df["health_index"].iloc[-1]
print(f"\nHEALTH INDEX          : {hi_now:.1f} / 100   ", end="")
if hi_now >= 80:   print("(HEALTHY)")
elif hi_now >= 60: print("(WATCH)")
else:              print("(ACTION REQUIRED)")

print(f"Current Rf / Critical : {last['Rf']:.6f} / {Rf_CRIT:.6f}")
print(f"Current dP / Critical : {last['delta_P_kPa']:.1f} / {dP_CRIT:.1f} kPa")
print(f"Efficiency            : {last['efficiency_pct']:.1f} %")
print(f"RUL (to critical Rf)  : approx {days_to_crit:.0f} days")
print(f"7-day status          : {status_7d}")
print(f"30-day status         : {status_30d}")

print("\nTOP RECOMMENDATIONS")
if hi_now < 40 or days_to_crit < 10:
    print("  1. Schedule cleaning within 7 days")
    print("  2. Review water-treatment set-points (hardness)")
    print("  3. Increase dP & U sampling to daily")
elif hi_now < 70 or days_to_crit < 30:
    print("  1. Plan cleaning in next maintenance window")
    print("  2. Consider mild velocity increase if pump allows")
    print("  3. Evaluate nanofluid trial for tube-side boost")
else:
    print("  1. Continue normal operation")
    print("  2. Keep water chemistry within target band")
    print("  3. Re-evaluate health index weekly")

print("\nNANOFLUID QUICK WIN (from Section 11)")
print("  CuO or Hybrid Al2O3-CuO at 1-2 vol% can raise tube-side h by ~5-12 %")
print("  -> higher U and delayed onset of critical fouling")

print("\nECONOMICS")
print(f"  Suggested cleaning interval under current assumptions: approx {best_iv} days")
print("=" * 78)
print("Dashboard complete — all modules integrated.")


## 17. Real CSV Data Ingestion

Load plant historian / SCADA exports. Expected columns (flexible mapping supported):

`timestamp, asset_id, operating_hours, water_hardness, velocity, T_hot_in, Rf, U, Q_kW, efficiency_pct, delta_P_kPa`

A sample CSV is written automatically so the pipeline is runnable out-of-the-box. Replace the path with your real file.


In [ ]:
print("=== CSV INGESTION MODULE ===")

sample_path = "hx_fleet_sample.csv"
np.random.seed(7)
rows = []
for asset, base_hard, base_vel in [("HX-A", 260, 1.5), ("HX-B", 320, 1.3), ("HX-C", 290, 1.6)]:
    for d in range(60):
        hrs = d * 24
        hard = base_hard + 30*np.sin(d/20) + np.random.randn()*5
        vel  = base_vel + np.random.randn()*0.08
        Th   = 140 + 6*np.sin(d/15) + np.random.randn()
        Rf_inf = 1.1e-4 * (hard/200)**1.1 * np.exp(0.01*(Th-100))
        tau_f  = 9000 * (max(vel,0.5)/1.5)**0.7 / (hard/200)**0.25
        Rf = max(0, Rf_inf*(1-np.exp(-hrs/tau_f)) + np.random.normal(0,2e-6))
        U  = 1/(1/920 + Rf)
        Q  = 650 * (U/920) * (0.99 + 0.02*np.random.randn())
        eff = Q/650*100
        dP = 25*(1 + 200*Rf + 5e5*Rf**2) + np.random.randn()*0.5
        rows.append({
            "timestamp": (pd.Timestamp("2026-03-01") + pd.Timedelta(days=d)).strftime("%Y-%m-%d"),
            "asset_id": asset,
            "operating_hours": hrs,
            "water_hardness": round(hard,1),
            "velocity": round(vel,3),
            "T_hot_in": round(Th,1),
            "Rf": Rf,
            "U": U,
            "Q_kW": Q,
            "efficiency_pct": eff,
            "delta_P_kPa": dP
        })
pd.DataFrame(rows).to_csv(sample_path, index=False)
print(f"Sample CSV written -> {sample_path}")

def load_hx_csv(path, asset=None):
    """Load HX sensor CSV. Optionally filter by asset_id."""
    df = pd.read_csv(path, parse_dates=["timestamp"])
    rename = {
        "date": "timestamp", "time": "timestamp",
        "hx_id": "asset_id", "equipment": "asset_id",
        "hours": "operating_hours", "hardness": "water_hardness",
        "vel": "velocity", "Tin_hot": "T_hot_in",
        "fouling": "Rf", "dP": "delta_P_kPa", "DP": "delta_P_kPa",
        "efficiency": "efficiency_pct", "eff": "efficiency_pct"
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})
    if asset and "asset_id" in df.columns:
        df = df[df["asset_id"] == asset].copy()
    df = df.sort_values("timestamp").reset_index(drop=True)
    return df

fleet_raw = load_hx_csv(sample_path)
print(f"Loaded {len(fleet_raw)} rows, assets: {sorted(fleet_raw['asset_id'].unique())}")
print(fleet_raw.head(3).to_string(index=False))
print("\nTo use your own data:  fleet_raw = load_hx_csv('/path/to/your_export.csv')")


## 18. Multi-HX Fleet Monitoring

Monitor several heat exchangers side-by-side: health index, Rf trend, RUL and priority ranking for maintenance planning.


In [ ]:
print("=== MULTI-HX FLEET MONITORING ===")

Rf_CRIT_F = 1.8e-4
dP_CRIT_F = 55.0

fleet_summary = []
for asset, grp in fleet_raw.groupby("asset_id"):
    g = grp.sort_values("timestamp")
    last_row = g.iloc[-1]
    rf_s  = np.clip(1 - last_row["Rf"] / Rf_CRIT_F, 0, 1)
    dp_s  = np.clip(1 - last_row["delta_P_kPa"] / dP_CRIT_F, 0, 1)
    ef_s  = np.clip((last_row["efficiency_pct"] - 70) / 30, 0, 1)
    hi    = (0.4*rf_s + 0.3*dp_s + 0.3*ef_s) * 100
    recent = g.tail(20)
    if len(recent) >= 5:
        slope = np.polyfit(recent["operating_hours"], recent["Rf"], 1)[0]
        rul_d = max(0, (Rf_CRIT_F - last_row["Rf"]) / slope / 24) if slope > 1e-12 else 999
    else:
        rul_d = 999
    status = "GREEN" if hi >= 80 else ("YELLOW" if hi >= 55 else "RED")
    fleet_summary.append({
        "asset_id": asset,
        "health_index": round(hi, 1),
        "Rf": last_row["Rf"],
        "delta_P_kPa": last_row["delta_P_kPa"],
        "efficiency_pct": last_row["efficiency_pct"],
        "RUL_days": round(rul_d, 0),
        "status": status,
        "last_timestamp": last_row["timestamp"]
    })

fleet_df = pd.DataFrame(fleet_summary).sort_values("health_index")
print(fleet_df.to_string(index=False))

print("\n--- Maintenance Priority (worst first) ---")
for _, r in fleet_df.iterrows():
    print(f"  {r['status']:6s}  {r['asset_id']}  HI={r['health_index']:.0f}  RUL~{r['RUL_days']:.0f}d  Rf={r['Rf']:.2e}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors_map = {"GREEN": "seagreen", "YELLOW": "orange", "RED": "crimson"}

ax = axes[0]
for asset, grp in fleet_raw.groupby("asset_id"):
    ax.plot(grp["timestamp"], grp["Rf"]*1e4, label=asset, lw=1.8)
ax.axhline(Rf_CRIT_F*1e4, color="r", ls="--", alpha=0.7)
ax.set_ylabel("Rf x 10^4"); ax.set_title("Fleet Fouling Trends"); ax.legend()

ax = axes[1]
ax.bar(fleet_df["asset_id"], fleet_df["health_index"],
       color=[colors_map[s] for s in fleet_df["status"]], edgecolor="k")
ax.axhline(80, color="g", ls=":"); ax.axhline(55, color="orange", ls=":")
ax.set_ylabel("Health Index"); ax.set_title("Fleet Health Snapshot"); ax.set_ylim(0, 105)

ax = axes[2]
ax.bar(fleet_df["asset_id"], fleet_df["RUL_days"],
       color=[colors_map[s] for s in fleet_df["status"]], edgecolor="k")
ax.set_ylabel("RUL [days]"); ax.set_title("Remaining Useful Life")
plt.tight_layout()
plt.show()


### Multi-HX Fleet Monitoring

![Multi-HX Fleet Monitoring](09_fleet_monitoring.png)

*Fouling trends, health-index snapshot and remaining useful life for the fleet.*


## 19. PDF Report Export

Generate a professional multi-page PDF report for management / maintenance planning. Saved under `/home/workdir/artifacts/`.


In [ ]:
print("=== PDF REPORT EXPORT ===")

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
                                 TableStyle, Image, HRFlowable)
from reportlab.lib.enums import TA_CENTER
import io

report_path = "HX_Predictive_Maintenance_Report.pdf"

def fig_to_image(fig, width=16*cm, height=9*cm):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    buf.seek(0)
    return Image(buf, width=width, height=height)

doc = SimpleDocTemplate(report_path, pagesize=A4,
                        rightMargin=1.5*cm, leftMargin=1.5*cm,
                        topMargin=1.5*cm, bottomMargin=1.5*cm)
styles = getSampleStyleSheet()
styles.add(ParagraphStyle(name="TitleCenter", parent=styles["Title"], alignment=TA_CENTER, fontSize=16))
styles.add(ParagraphStyle(name="Section", parent=styles["Heading2"], spaceBefore=12, textColor=colors.HexColor("#1a365d")))
styles.add(ParagraphStyle(name="Body", parent=styles["Normal"], fontSize=10, leading=14))
styles.add(ParagraphStyle(name="Small", parent=styles["Normal"], fontSize=8, textColor=colors.grey))

story = []
story.append(Paragraph("Heat Exchanger Predictive Maintenance Report", styles["TitleCenter"]))
story.append(Spacer(1, 6))
story.append(HRFlowable(width="100%", thickness=2, color=colors.HexColor("#2b6cb0")))
story.append(Spacer(1, 8))
story.append(Paragraph("Generated: " + datetime.now().strftime("%Y-%m-%d %H:%M"), styles["Body"]))
story.append(Paragraph("Assets: Shell-Tube fleet (HX-A, HX-B, HX-C) + detailed single-unit analysis", styles["Body"]))
story.append(Spacer(1, 16))

story.append(Paragraph("1. Executive Summary", styles["Section"]))
hi_val = float(sensor_df["health_index"].iloc[-1]) if "health_index" in sensor_df.columns else 75.0
summary_text = (
    "Primary unit health index: <b>{:.1f}/100</b>. "
    "Estimated RUL to critical fouling: <b>approx {:.0f} days</b>. "
    "7-day status: <b>{}</b>, 30-day status: <b>{}</b>. "
    "Fleet worst unit should be prioritised for the next cleaning window. "
    "Suggested cleaning interval under current economic assumptions: <b>approx {} days</b>."
).format(hi_val, days_to_crit, status_7d, status_30d, best_iv)
story.append(Paragraph(summary_text, styles["Body"]))
story.append(Spacer(1, 10))

story.append(Paragraph("2. Fleet Status Snapshot", styles["Section"]))
fleet_table_data = [["Asset", "Health", "Rf", "dP [kPa]", "Eff %", "RUL [d]", "Status"]]
for _, r in fleet_df.iterrows():
    fleet_table_data.append([
        str(r["asset_id"]), "{:.0f}".format(r["health_index"]), "{:.2e}".format(r["Rf"]),
        "{:.1f}".format(r["delta_P_kPa"]), "{:.1f}".format(r["efficiency_pct"]),
        "{:.0f}".format(r["RUL_days"]), str(r["status"])
    ])
t = Table(fleet_table_data, colWidths=[2.2*cm, 1.8*cm, 2.4*cm, 2.2*cm, 1.8*cm, 2*cm, 2*cm])
t.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2b6cb0")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTSIZE", (0, 0), (-1, -1), 8),
    ("ALIGN", (1, 0), (-1, -1), "CENTER"),
    ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#ebf8ff")]),
    ("TOPPADDING", (0, 0), (-1, -1), 4),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
]))
story.append(t)
story.append(Spacer(1, 14))

story.append(Paragraph("3. Primary Unit - Key Indicators", styles["Section"]))
kpi_data = [
    ["Metric", "Value", "Threshold / Note"],
    ["Fouling Rf", "{:.6f} m2K/W".format(last["Rf"]), "Critical {:.1e}".format(Rf_CRIT)],
    ["Pressure drop", "{:.1f} kPa".format(last["delta_P_kPa"]), "Critical {:.0f} kPa".format(dP_CRIT)],
    ["Efficiency", "{:.1f} %".format(last["efficiency_pct"]), "Target > 85 %"],
    ["Overall U", "{:.1f} W/m2K".format(last["U"]), "-"],
    ["RUL", "approx {:.0f} days".format(days_to_crit), "Action if < 30 d"],
]
t2 = Table(kpi_data, colWidths=[4*cm, 5*cm, 5*cm])
t2.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2b6cb0")),
    ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
    ("FONTSIZE", (0, 0), (-1, -1), 9),
    ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f7fafc")]),
    ("TOPPADDING", (0, 0), (-1, -1), 5),
    ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
]))
story.append(t2)
story.append(Spacer(1, 14))

story.append(Paragraph("4. Recommended Actions", styles["Section"]))
if days_to_crit < 10 or status_now == "RED":
    recs = [
        "Schedule chemical/mechanical cleaning within 7 days.",
        "Increase monitoring frequency (daily dP and U).",
        "Review water-treatment hardness and pH set-points.",
        "Consider temporary load reduction until cleaned."
    ]
elif days_to_crit < 30:
    recs = [
        "Plan cleaning in the next maintenance window (within 2-4 weeks).",
        "Evaluate mild velocity increase if pump capacity allows.",
        "Consider a controlled nanofluid trial (CuO or hybrid 1 vol%) for tube-side boost.",
        "Keep weekly health-index review."
    ]
else:
    recs = [
        "Continue normal operation.",
        "Maintain water chemistry within target band.",
        "Re-evaluate health index and RUL weekly.",
        "Optional: nanofluid evaluation for long-term efficiency gain."
    ]
for i, rec in enumerate(recs, 1):
    story.append(Paragraph("{}. {}".format(i, rec), styles["Body"]))

story.append(Spacer(1, 16))
story.append(Paragraph("5. Charts", styles["Section"]))

fig1, axes1 = plt.subplots(1, 2, figsize=(9, 3.2))
axes1[0].plot(sensor_df["date"], sensor_df["Rf"]*1e4, "b-", lw=1.5)
axes1[0].axhline(Rf_CRIT*1e4, color="r", ls="--")
axes1[0].set_title("Fouling Rf"); axes1[0].set_ylabel("Rf x 10^4")
axes1[0].tick_params(axis="x", rotation=30)
axes1[1].plot(sensor_df["date"], sensor_df["efficiency_pct"], "g-", lw=1.5)
axes1[1].set_title("Efficiency %"); axes1[1].tick_params(axis="x", rotation=30)
plt.tight_layout()
story.append(fig_to_image(fig1, width=16*cm, height=6*cm))
plt.close(fig1)

fig2, ax2 = plt.subplots(figsize=(7, 3))
ax2.bar(fleet_df["asset_id"], fleet_df["health_index"],
        color=[colors_map.get(s, "steelblue") for s in fleet_df["status"]], edgecolor="k")
ax2.axhline(80, color="g", ls=":"); ax2.axhline(55, color="orange", ls=":")
ax2.set_ylim(0, 105); ax2.set_ylabel("Health Index"); ax2.set_title("Fleet Health Index")
plt.tight_layout()
story.append(fig_to_image(fig2, width=14*cm, height=6*cm))
plt.close(fig2)

story.append(Spacer(1, 12))
story.append(HRFlowable(width="100%", thickness=1, color=colors.grey))
story.append(Paragraph(
    "This report is generated automatically from the Heat Exchanger Predictive Maintenance Suite. "
    "Thresholds and economic assumptions can be adjusted in the notebook source.",
    styles["Small"]
))

doc.build(story)
print("PDF report saved -> " + report_path)
print("Open the file to view the multi-page management report.")


## 21. Additional Engineering Features (v2.1)

New modules added:

| Feature | Description |
|---------|-------------|
| Pressure Drop | Tube-side & shell-side ΔP (Darcy + Kern method) |
| LMTD Correction F | Multipass correction factor for 1-2 exchangers |
| Capital Cost | Approximate purchased equipment cost (Hall-style) |
| Axial T Profiles | Hot & cold fluid temperature along tube length |
| Vibration Risk | Simple flow-induced vibration screening |
| Maintenance Calendar | Next recommended cleaning windows |
| Excel Export | One-click summary workbook |


### 21.1 Pressure Drop (Tube-side & Shell-side)


In [ ]:
print("=== PRESSURE DROP MODULE ===")

def tube_side_dp(m_dot, rho, mu, di, L, n_tubes, n_passes=2, roughness=4.5e-5):
    area = n_tubes * np.pi * (di/2)**2 / n_passes
    vel = m_dot / (rho * area)
    Re = rho * vel * di / mu
    f = (-1.8 * np.log10((roughness/di/3.7)**1.11 + 6.9/Re))**(-2)
    dp_friction = f * (L * n_passes / di) * (rho * vel**2 / 2)
    dp_return = 1.5 * n_passes * (rho * vel**2 / 2)
    return dp_friction + dp_return, vel, Re

def shell_side_dp_kern(m_dot, rho, mu, Ds, do, B, n_baffles, pitch_ratio=1.25):
    clearance = pitch_ratio * do - do
    As = (Ds * clearance * B) / (pitch_ratio * do)
    Gs = m_dot / As
    De = 4 * (0.86 * (pitch_ratio*do)**2 - 0.5*np.pi*do**2/2) / (np.pi * do)
    Re_s = De * Gs / mu
    f_s = 0.36 * Re_s**(-0.18) if Re_s > 0 else 0.5
    N_cross = n_baffles + 1
    dp = f_s * Gs**2 * (N_cross) * Ds / (2 * rho * De)
    return dp, Gs, Re_s

rho, mu = 980.0, 4e-4
dp_t, vel_t, Re_t = tube_side_dp(2.5, rho, mu, hx_shell.di, hx_shell.L, hx_shell.n_tubes)
dp_s, Gs, Re_s = shell_side_dp_kern(3.0, rho, mu, hx_shell.Ds, hx_shell.do, hx_shell.B, hx_shell.n_baffles)

print(f"Tube-side  ΔP : {dp_t/1000:.1f} kPa   |  velocity {vel_t:.2f} m/s  |  Re {Re_t:.0f}")
print(f"Shell-side ΔP : {dp_s/1000:.1f} kPa   |  Gs {Gs:.1f} kg/m²s |  Re_s {Re_s:.0f}")
print(f"Total approx  : {(dp_t+dp_s)/1000:.1f} kPa")
if vel_t > 2.5:
    print("Warning: Tube velocity high – erosion risk")
elif vel_t < 0.9:
    print("Warning: Tube velocity low – fouling risk")
else:
    print("OK: Tube velocity in preferred range (0.9–2.5 m/s)")


### 21.2 LMTD Correction Factor F (1 shell pass – 2 tube passes)


In [ ]:
print("=== LMTD CORRECTION FACTOR F ===")

def lmtd_correction_F(T1, T2, t1, t2):
    dT1 = T1 - t2
    dT2 = T2 - t1
    if abs(dT1 - dT2) < 1e-6:
        LMTD = dT1
    else:
        LMTD = (dT1 - dT2) / np.log(dT1 / dT2)
    P = (t2 - t1) / (T1 - t1)
    R = (T1 - T2) / (t2 - t1)
    if abs(R - 1) < 1e-4:
        F = (np.sqrt(2) * P / ((1-P) * np.log((2-P*(2-np.sqrt(2)))/(2-P*(2+np.sqrt(2))))))
    else:
        num = np.sqrt(R**2 + 1) * np.log((1-P)/(1-R*P))
        den = (R-1) * np.log( (2-P*(R+1-np.sqrt(R**2+1))) / (2-P*(R+1+np.sqrt(R**2+1))) )
        F = num / den if abs(den) > 1e-12 else 1.0
    F = np.clip(F, 0.5, 1.0)
    return F, LMTD, P, R

res = hx_shell.steady_state(2.5, 4180, 120, 3.0, 4180, 25, U=800)
F, LMTD, P, R = lmtd_correction_F(120, res['Th_out'], 25, res['Tc_out'])
print(f"Pure counter-flow LMTD : {LMTD:.2f} K")
print(f"Correction factor F    : {F:.3f}")
print(f"Corrected LMTD (F*LMTD): {F*LMTD:.2f} K")
print(f"P (effectiveness)      : {P:.3f}   R (capacity ratio): {R:.3f}")
if F < 0.75:
    print("Warning: F is low – temperature cross risk / consider more shell passes")
else:
    print("OK: F is acceptable for 1-2 configuration")


### 21.3 Approximate Capital Cost (Purchased Equipment)


In [ ]:
print("=== CAPITAL COST ESTIMATION (Hall-style) ===")

def hx_capital_cost(A, material='CS', pressure_bar=10, year=2026):
    C0 = 10000 * (A / 10)**0.68
    mat_factor = {'CS': 1.0, 'SS304': 2.5, 'SS316': 3.0, 'Ti': 8.0, 'Hastelloy': 12.0}.get(material, 1.0)
    p_factor = 1.0 + 0.05 * max(0, pressure_bar - 10)
    infl = 1.55
    return C0 * mat_factor * p_factor * infl

for mat in ['CS', 'SS304', 'SS316', 'Ti']:
    cost = hx_capital_cost(hx_shell.A, material=mat)
    print(f"  {mat:10s}  Area {hx_shell.A:.1f} m2  ->  approx ${cost:,.0f}")

print("\nNote: Order-of-magnitude only. Use detailed vendor quotes for final decisions.")


### 21.4 Axial Temperature Profiles (Counter-flow)


In [ ]:
print("=== AXIAL TEMPERATURE PROFILES ===")

def axial_profiles(hx, m_h, cp_h, Th_in, m_c, cp_c, Tc_in, U, n=50):
    C_h = m_h * cp_h
    C_c = m_c * cp_c
    C_min = min(C_h, C_c)
    NTU = U * hx.A / C_min
    Cr = C_min / max(C_h, C_c)
    eps = hx.effectiveness_ntu(NTU, Cr, 'counter')
    Q = eps * C_min * (Th_in - Tc_in)
    Th_out = Th_in - Q / C_h
    Tc_out = Tc_in + Q / C_c
    x = np.linspace(0, 1, n)
    if abs(C_h - C_c) < 1e-6:
        Th = Th_in - (Th_in - Th_out) * x
        Tc = Tc_out - (Tc_out - Tc_in) * x
    else:
        alpha = (1 - Cr) * NTU
        Th = Th_in - (Th_in - Th_out) * (1 - np.exp(-alpha * x)) / (1 - np.exp(-alpha))
        Tc = Tc_out - (Tc_out - Tc_in) * (1 - np.exp(-alpha * x)) / (1 - np.exp(-alpha))
    return x, Th, Tc, Th_out, Tc_out

x, Th, Tc, Th_o, Tc_o = axial_profiles(hx_shell, 2.5, 4180, 120, 3.0, 4180, 25, 800)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, Th, 'r-', lw=2.5, label=f'Hot fluid (out={Th_o:.1f} C)')
ax.plot(x, Tc, 'b-', lw=2.5, label=f'Cold fluid (out={Tc_o:.1f} C)')
ax.fill_between(x, Th, Tc, alpha=0.15, color='orange')
ax.set_xlabel('Normalised length (0 = hot inlet)')
ax.set_ylabel('Temperature [C]')
ax.set_title('Axial Temperature Profiles — Counter-flow Shell-Tube')
ax.legend()
plt.tight_layout()
fig.savefig('10_axial_temperature_profiles.png', dpi=140, bbox_inches='tight')
plt.show()
print("Saved 10_axial_temperature_profiles.png")


### 21.5 Simple Flow-Induced Vibration Screening


In [ ]:
print("=== FLOW-INDUCED VIBRATION SCREENING ===")

def vibration_screen(vel_shell, do, pitch, span, rho_shell=980):
    V_crit_approx = 1.8 * (do / 0.025) * (0.6 / span)**0.5
    ratio = vel_shell / V_crit_approx
    if ratio < 0.5:
        risk = "LOW"
    elif ratio < 0.8:
        risk = "MODERATE – check baffle spacing"
    else:
        risk = "HIGH – redesign recommended"
    return V_crit_approx, ratio, risk

vel_shell_approx = Gs / rho if 'Gs' in dir() else 0.8
Vcrit, ratio, risk = vibration_screen(vel_shell_approx, hx_shell.do, 1.25*hx_shell.do, hx_shell.B)
print(f"Approx shell cross-flow velocity : {vel_shell_approx:.2f} m/s")
print(f"Rough critical velocity          : {Vcrit:.2f} m/s")
print(f"Velocity ratio                   : {ratio:.2f}")
print(f"Risk assessment                  : {risk}")
print("(Full TEMA / ASME vibration analysis requires detailed modal data.)")


### 21.6 Recommended Maintenance Calendar


In [ ]:
print("=== MAINTENANCE CALENDAR (next 12 months) ===")

from datetime import datetime, timedelta

today = datetime(2026, 11, 18)
try:
    interval = best_iv
except:
    interval = 60

try:
    rul = days_to_crit
except:
    rul = 90

print(f"Suggested cleaning interval : every {interval} days")
print(f"Estimated RUL (current unit): ~ {rul:.0f} days\n")

events = []
for i in range(1, 7):
    d = today + timedelta(days=int(i * interval))
    events.append((d.strftime('%Y-%m-%d'), f"Planned cleaning #{i}", "Maintenance"))

insp = today + timedelta(days=int(interval/2))
events.append((insp.strftime('%Y-%m-%d'), "Visual / dP inspection", "Inspection"))
events = sorted(events, key=lambda x: x[0])

print(f"{'Date':12s}  {'Activity':30s}  {'Type'}")
print("-" * 55)
for date, act, typ in events:
    print(f"{date:12s}  {act:30s}  {typ}")


### 21.7 Excel Summary Export


In [ ]:
print("=== EXCEL SUMMARY EXPORT ===")

summary = {
    "Metric": [
        "Heat Duty Q (kW)", "Effectiveness eps", "Overall U (W/m2K)", "Area (m2)",
        "Tube-side dP (kPa)", "Shell-side dP (kPa)", "LMTD Correction F",
        "Capital Cost CS (USD)", "Health Index", "RUL (days)",
        "Suggested Cleaning Interval (days)"
    ],
    "Value": [
        round(res.get('Q', 0)/1000, 1),
        round(res.get('eps', 0), 3),
        round(res.get('U', 0), 1),
        round(hx_shell.A, 2),
        round(dp_t/1000, 1),
        round(dp_s/1000, 1),
        round(F, 3),
        round(hx_capital_cost(hx_shell.A, 'CS'), 0),
        round(float(sensor_df['health_index'].iloc[-1]) if 'health_index' in sensor_df.columns else 75, 1),
        round(float(days_to_crit) if 'days_to_crit' in dir() else 90, 0),
        interval
    ]
}
df_export = pd.DataFrame(summary)
excel_path = "HX_Summary_Export.xlsx"
df_export.to_excel(excel_path, index=False)
print(f"Excel summary written -> {excel_path}")
print(df_export.to_string(index=False))


## 20. How to use with real plant data

1. Export daily (or hourly) historian data to CSV with the columns listed in Section 17.
2. Call `fleet_raw = load_hx_csv("your_file.csv")`.
3. Re-run fleet monitoring and the PDF export cells.
4. Adjust `Rf_CRIT`, `dP_CRIT`, energy price and cleaning cost to match your site.
5. For a single critical exchanger, filter: `df = load_hx_csv(path, asset="HX-A")` and feed into the earlier ML / RUL pipeline.


## 21. Additional Engineering Modules (v2.1)

New capabilities added in this release:

| Module | What it does |
|--------|--------------|
| Pressure Drop | Tube-side + simplified shell-side ΔP |
| Pumping Power | Hydraulic power from flow & ΔP (incl. nanofluid viscosity effect) |
| CAPEX Estimate | Area-based equipment cost (Hall-style correlation) |
| Exergy Efficiency | Second-law performance indicator |
| TEMA Helper | Quick shell type / layout reminder |
| Multi-objective snapshot | Q vs ΔP vs Cost trade-off view |


In [ ]:
print("=== 21. PRESSURE DROP · PUMPING POWER · CAPEX · EXERGY ===")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Helpers ----
def tube_side_dP(m_dot, rho, mu, di, L, n_tubes, n_passes=2, roughness=4.5e-5):
    """Darcy–Weisbach tube-side pressure drop (Pa)."""
    A_flow = n_tubes * np.pi * (di/2)**2 / n_passes
    v = m_dot / (rho * A_flow)
    Re = rho * v * di / mu
    # Haaland friction factor
    f = (-1.8 * np.log10((roughness/di/3.7)**1.11 + 6.9/Re))**(-2) if Re > 0 else 0.03
    dP_friction = f * (L * n_passes / di) * (rho * v**2 / 2)
    dP_return = 1.5 * n_passes * (rho * v**2 / 2)   # return losses
    return dP_friction + dP_return, v, Re

def shell_side_dP_simple(m_dot, rho, mu, Ds, B, do, n_tubes, L):
    """Very simplified shell-side ΔP (Kern-style order-of-magnitude)."""
    # Cross-flow area approximation
    As = Ds * B * (1 - 0.5)          # rough clearance factor
    v_s = m_dot / (rho * max(As, 1e-4))
    Re_s = rho * v_s * do / mu
    f_s = 1.73 * Re_s**(-0.2) if Re_s > 0 else 0.3
    Nb = max(int(L / B) - 1, 1)
    dP = f_s * (Ds / do) * Nb * (rho * v_s**2 / 2) * 1.2
    return dP, v_s

def pumping_power(m_dot, dP, eta_pump=0.7):
    """Hydraulic + shaft power (W)."""
    P_hyd = m_dot * dP / 1000.0          # rough (m³/s * Pa) ≈ W when rho~1000
    # more correctly: V_dot * dP
    return P_hyd / eta_pump

def capex_hall(A, material_factor=1.0, pressure_factor=1.0, year_factor=1.3):
    """Simple Hall-style cost correlation (USD, order-of-magnitude)."""
    # Base: C = a + b * A**c  (carbon steel, moderate pressure)
    C = 12000 + 1800 * (A ** 0.81)
    return C * material_factor * pressure_factor * year_factor

def exergy_efficiency(Q, Th_in, Th_out, Tc_in, Tc_out, T0=298.15):
    """Simplified exergy efficiency for a heat exchanger."""
    # Average temperatures (K)
    Th_m = 0.5 * (Th_in + Th_out) + 273.15
    Tc_m = 0.5 * (Tc_in + Tc_out) + 273.15
    Ex_h = Q * (1 - T0 / Th_m)
    Ex_c = Q * (1 - T0 / Tc_m)
    if Ex_h <= 0:
        return np.nan
    return Ex_c / Ex_h

# ---- Evaluate for the three units under common conditions ----
rho, mu = 980.0, 4e-4          # hot water-ish
m_h, m_c = 2.5, 3.0
cp = 4180
Th_in, Tc_in = 120.0, 25.0

rows = []
for name, hx, n_pass in [
    ("Shell-Tube", hx_shell, 2),
    ("Plate", hx_plate, 1),
    ("Finned-Tube", hx_fin, 1),
]:
    res = hx.steady_state(m_h, cp, Th_in, m_c, cp, Tc_in)
    # Geometry defaults for ΔP
    if name == "Shell-Tube":
        di, L, n_t, Ds, B = hx.di, hx.L, hx.n_tubes, hx.Ds, hx.B
        dP_t, v_t, Re_t = tube_side_dP(m_h, rho, mu, di, L, n_t, n_pass)
        dP_s, v_s = shell_side_dP_simple(m_c, rho, mu, Ds, B, hx.do, n_t, L)
        dP_tot = dP_t + dP_s
    else:
        # Approximate for plate / finned
        di = 0.02 if name == "Plate" else 0.022
        L = 1.5 if name == "Plate" else hx.L
        n_t = 40 if name == "Plate" else hx.n_tubes
        dP_t, v_t, Re_t = tube_side_dP(m_h, rho, mu, di, L, n_t, 1)
        dP_s = dP_t * 0.6
        dP_tot = dP_t + dP_s
        v_s = v_t * 0.7

    P_pump = pumping_power(m_h, dP_tot) + pumping_power(m_c, dP_tot * 0.8)
    cost = capex_hall(res["A"], material_factor=1.0 if name != "Plate" else 1.25)
    eta_ex = exergy_efficiency(res["Q"], Th_in, res["Th_out"], Tc_in, res["Tc_out"])

    rows.append({
        "HX": name,
        "Q_kW": res["Q"]/1000,
        "eps": res["eps"],
        "A_m2": res["A"],
        "dP_kPa": dP_tot/1000,
        "v_tube_m_s": v_t,
        "Pump_kW": P_pump/1000,
        "CAPEX_USD": cost,
        "Exergy_eff": eta_ex,
        "Q_per_cost": (res["Q"]/1000) / (cost/1000),
    })

df_extra = pd.DataFrame(rows).set_index("HX")
print(df_extra.round(3).to_string())
print("\nNotes: ΔP and CAPEX are order-of-magnitude engineering estimates.")


In [ ]:
print("=== MULTI-OBJECTIVE SNAPSHOT (Q vs ΔP vs CAPEX) ===")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

colors = ["steelblue", "seagreen", "coral"]

# 1. Heat duty vs CAPEX
axes[0].scatter(df_extra["CAPEX_USD"]/1000, df_extra["Q_kW"],
                s=180, c=colors, edgecolors="k", zorder=3)
for i, name in enumerate(df_extra.index):
    axes[0].annotate(name, (df_extra["CAPEX_USD"].iloc[i]/1000, df_extra["Q_kW"].iloc[i]),
                     textcoords="offset points", xytext=(6, 4), fontsize=9)
axes[0].set_xlabel("CAPEX [kUSD]")
axes[0].set_ylabel("Heat Duty [kW]")
axes[0].set_title("Q vs Equipment Cost")

# 2. Heat duty vs Pressure drop
axes[1].scatter(df_extra["dP_kPa"], df_extra["Q_kW"],
                s=180, c=colors, edgecolors="k", zorder=3)
for i, name in enumerate(df_extra.index):
    axes[1].annotate(name, (df_extra["dP_kPa"].iloc[i], df_extra["Q_kW"].iloc[i]),
                     textcoords="offset points", xytext=(6, 4), fontsize=9)
axes[1].set_xlabel("Total ΔP [kPa]")
axes[1].set_ylabel("Heat Duty [kW]")
axes[1].set_title("Q vs Pressure Drop")

# 3. Exergy efficiency bar
axes[2].bar(df_extra.index, df_extra["Exergy_eff"], color=colors, edgecolor="k")
axes[2].set_ylabel("Exergy Efficiency")
axes[2].set_title("Second-Law Performance")
axes[2].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("10_multiobjective_snapshot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 10_multiobjective_snapshot.png")


![Multi-Objective Snapshot](10_multiobjective_snapshot.png)

*Heat duty versus CAPEX and pressure drop, plus exergy efficiency comparison.*


### TEMA Designation Helper

Quick reference for common TEMA shell types and when to use them.


In [ ]:
print("=== TEMA SHELL-TYPE HELPER ===")

tema = {
    "E": {"name": "One-pass shell", "use": "Most common; single shell pass, baffles", "note": "Default choice for many duties"},
    "F": {"name": "Two-pass shell with longitudinal baffle", "use": "When pure counterflow is needed", "note": "Thermal leakage across baffle possible"},
    "G": {"name": "Split flow", "use": "Horizontal thermosyphon reboilers", "note": "Lower ΔP than E"},
    "H": {"name": "Double split flow", "use": "Large thermosyphon reboilers", "note": "Even lower ΔP"},
    "J": {"name": "Divided flow", "use": "Very low shell-side ΔP", "note": "Two inlets / one outlet or vice-versa"},
    "K": {"name": "Kettle type", "use": "Reboilers, vaporizers", "note": "Large disengagement space"},
    "X": {"name": "Cross flow", "use": "Very low ΔP, condensers", "note": "No baffles; pure crossflow"},
}

print(f"{'Type':<6} {'Name':<40} {'Typical use'}")
print("-" * 90)
for k, v in tema.items():
    print(f"{k:<6} {v['name']:<40} {v['use']}")

print("\nFront/Rear head types (common):")
print("  A/L  – Channel with removable cover (cleaning)")
print("  B/M  – Bonnet (cheaper, less accessible)")
print("  C/N  – Channel integral with tube-sheet")
print("  S    – Floating head with backing device (thermal expansion)")
print("  T    – Pull-through floating head")
print("  U    – U-tube bundle (no floating head needed)")

print("\nExample designation:  AEL  =  Channel (A) + One-pass shell (E) + Fixed tube-sheet rear (L)")


### Pumping Power Impact of Nanofluids

Higher viscosity of nanofluids increases ΔP and pumping power. This cell quantifies the trade-off.


In [ ]:
print("=== NANOFLUID PUMPING POWER IMPACT ===")

# Re-use earlier nanofluid property ideas (Brinkman viscosity)
def brinkman_mu(mu_bf, phi):
    return mu_bf / (1 - phi)**2.5

mu_water = 8.9e-4
phi_list = [0.0, 0.005, 0.01, 0.02, 0.03]
base_dP = 35e3   # Pa reference clean water
base_power = pumping_power(2.5, base_dP)

print(f"{'phi %':>8}  {'mu_ratio':>10}  {'dP_ratio':>10}  {'Pump_kW':>10}  {'ΔP vs water':>12}")
pump_rows = []
for phi in phi_list:
    mu_r = brinkman_mu(mu_water, phi) / mu_water
    # Approximate ΔP scales with mu^0.25 (turbulent) to mu^1 (laminar); use 0.4 as compromise
    dP_r = mu_r ** 0.4
    P = pumping_power(2.5, base_dP * dP_r) / 1000
    pump_rows.append({"phi_%": phi*100, "mu_ratio": mu_r, "dP_ratio": dP_r, "Pump_kW": P})
    print(f"{phi*100:8.1f}  {mu_r:10.3f}  {dP_r:10.3f}  {P:10.3f}  {100*(dP_r-1):+10.1f} %")

df_pump = pd.DataFrame(pump_rows)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_pump["phi_%"], df_pump["Pump_kW"], "o-", lw=2, color="crimson", label="Pumping power")
ax.set_xlabel("Particle volume fraction φ [%]")
ax.set_ylabel("Pumping power [kW]")
ax.set_title("Nanofluid Viscosity Effect on Pumping Power")
ax.legend()
plt.tight_layout()
plt.savefig("11_nanofluid_pumping.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 11_nanofluid_pumping.png")


![Nanofluid Pumping Power](11_nanofluid_pumping.png)

*Effect of nanoparticle volume fraction on required pumping power.*


In [ ]:
print("=" * 70)
print("NEW MODULES SUMMARY (v2.1)")
print("=" * 70)
print(df_extra[["Q_kW", "dP_kPa", "Pump_kW", "CAPEX_USD", "Exergy_eff"]].round(2).to_string())
print("\nFiles written: 10_multiobjective_snapshot.png , 11_nanofluid_pumping.png")
print("TEMA helper and pressure-drop / CAPEX / exergy routines are now part of the suite.")
print("=" * 70)


## 22. More Engineering Modules (v2.2)

| Module | Description |
|--------|-------------|
| LMTD Correction Factor F | Multipass (1-2, 2-4) F-charts approximation |
| Axial Temperature Profiles | Simple 1-D counter / parallel flow profiles |
| TEMA Fouling Factors | Recommended Rf values by fluid service |
| Q–Area Design Curve | Heat duty vs required area at fixed U |
| Tube-Sheet Layout Sketch | Simple 2-D tube pattern visualizer |
| Results CSV Export | One-click summary table for reports |
| Unit Converter | Common SI ↔ US customary conversions |


In [ ]:
print("=== 22.1 LMTD CORRECTION FACTOR F (multipass) ===")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def F_1_2(P, R):
    """Bowman approximation for 1 shell pass / 2 tube passes."""
    # Guard invalid regions
    P = np.clip(P, 1e-6, 0.999)
    if abs(R - 1) < 1e-6:
        return P / (1 - P) * (1 - (1 - P) / (1 + (1 - P) * np.log(1 - P) / P))
    num = np.sqrt(R**2 + 1) * np.log((1 - P) / (1 - R * P))
    den = (R - 1) * np.log( (2 - P*(R + 1 - np.sqrt(R**2+1))) /
                            (2 - P*(R + 1 + np.sqrt(R**2+1))) )
    return num / den if den != 0 else np.nan

# Design chart data
P_vals = np.linspace(0.05, 0.95, 40)
R_list = [0.2, 0.4, 0.6, 0.8, 1.0, 1.5, 2.0, 3.0, 4.0]

fig, ax = plt.subplots(figsize=(9, 6))
for R in R_list:
    F = [F_1_2(P, R) for P in P_vals]
    ax.plot(P_vals, F, lw=1.8, label=f"R = {R}")
ax.set_xlabel("P = (T_t,out − T_t,in) / (T_s,in − T_t,in)")
ax.set_ylabel("F (LMTD correction)")
ax.set_title("1 Shell-Pass / 2 Tube-Pass  —  F Chart (approx.)")
ax.set_ylim(0.5, 1.05)
ax.axhline(0.75, color="red", ls="--", alpha=0.6, label="F = 0.75 (typical min)")
ax.legend(ncol=2, fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("12_LMTD_F_chart.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 12_LMTD_F_chart.png")

# Example evaluation
P_ex, R_ex = 0.6, 0.8
print(f"Example: P={P_ex}, R={R_ex}  →  F ≈ {F_1_2(P_ex, R_ex):.3f}")


![12_LMTD_F_chart.png](12_LMTD_F_chart.png)

*LMTD correction factor F chart (1-2 multipass).*


In [ ]:
print("=== 22.2 AXIAL TEMPERATURE PROFILES ===")

def axial_profiles(Th_in, Tc_in, C_h, C_c, UA, n=80, arrangement="counter"):
    """Simple 1-D discretised temperature profiles."""
    x = np.linspace(0, 1, n)
    dx = 1.0 / (n - 1)
    Th = np.zeros(n); Tc = np.zeros(n)
    if arrangement == "counter":
        # Iterative shooting for counter-flow
        Th[0] = Th_in
        # Guess Tc at hot inlet end (x=0 is hot inlet, cold outlet)
        Tc0_guess = Tc_in + 30
        for _ in range(40):
            Tc[0] = Tc0_guess
            for i in range(n-1):
                dQ = UA * dx * (Th[i] - Tc[i])
                Th[i+1] = Th[i] - dQ / C_h
                Tc[i+1] = Tc[i] - dQ / C_c   # cold flows opposite
            # cold inlet is at x=1
            err = Tc[-1] - Tc_in
            Tc0_guess -= 0.5 * err
            if abs(err) < 0.05:
                break
        # Flip cold so plot shows both from hot-inlet side
        Tc_plot = Tc[::-1]
    else:  # parallel
        Th[0] = Th_in; Tc[0] = Tc_in
        for i in range(n-1):
            dQ = UA * dx * (Th[i] - Tc[i])
            Th[i+1] = Th[i] - dQ / C_h
            Tc[i+1] = Tc[i] + dQ / C_c
        Tc_plot = Tc
    return x, Th, Tc_plot

C_h, C_c = 2.5*4180, 3.0*4180
UA = 800 * 23.56
x, Th_c, Tc_c = axial_profiles(120, 25, C_h, C_c, UA, arrangement="counter")
x, Th_p, Tc_p = axial_profiles(120, 25, C_h, C_c, UA, arrangement="parallel")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(x, Th_c, "r-", lw=2, label="Hot fluid")
axes[0].plot(x, Tc_c, "b-", lw=2, label="Cold fluid")
axes[0].set_title("Counter-flow"); axes[0].set_xlabel("Dimensionless length x/L")
axes[0].set_ylabel("Temperature [°C]"); axes[0].legend()

axes[1].plot(x, Th_p, "r-", lw=2, label="Hot fluid")
axes[1].plot(x, Tc_p, "b-", lw=2, label="Cold fluid")
axes[1].set_title("Parallel-flow"); axes[1].set_xlabel("Dimensionless length x/L")
axes[1].set_ylabel("Temperature [°C]"); axes[1].legend()

plt.suptitle("Axial Temperature Profiles (same UA, C_h, C_c)", fontweight="bold")
plt.tight_layout()
plt.savefig("13_axial_temperature_profiles.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 13_axial_temperature_profiles.png")


![13_axial_temperature_profiles.png](13_axial_temperature_profiles.png)

*Axial temperature profiles — counter-flow vs parallel-flow.*


In [ ]:
print("=== 22.3 TEMA RECOMMENDED FOULING FACTORS ===")

fouling_table = [
    ("Sea water (≤ 50 °C)", 0.00009),
    ("Sea water (> 50 °C)", 0.00018),
    ("Brackish water", 0.00035),
    ("Cooling tower water (treated)", 0.00018),
    ("Cooling tower water (untreated)", 0.00035),
    ("River water", 0.00035),
    ("Distilled / boiler feed", 0.00009),
    ("Engine jacket water", 0.00018),
    ("Fuel oil (light)", 0.00035),
    ("Fuel oil (heavy)", 0.00088),
    ("Crude oil", 0.00035),
    ("Vegetable oils", 0.00053),
    ("Steam (oil-free)", 0.00009),
    ("Steam (with oil traces)", 0.00018),
    ("Organic vapours / solvents", 0.00018),
    ("Refrigerant liquids", 0.00018),
    ("Hydraulic fluid", 0.00018),
    ("Air / industrial gases", 0.00035),
]

df_rf = pd.DataFrame(fouling_table, columns=["Service", "Rf_m2K_W"])
df_rf["Rf_ft2hF_Btu"] = df_rf["Rf_m2K_W"] * 5.678   # approx conversion
print(df_rf.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(df_rf["Service"], df_rf["Rf_m2K_W"]*1e4, color="steelblue", edgecolor="k")
ax.set_xlabel("Rf × 10⁴  [m²K/W]")
ax.set_title("TEMA Recommended Fouling Factors (selected services)")
plt.tight_layout()
plt.savefig("14_TEMA_fouling_factors.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 14_TEMA_fouling_factors.png")


![14_TEMA_fouling_factors.png](14_TEMA_fouling_factors.png)

*TEMA recommended fouling factors by service.*


In [ ]:
print("=== 22.4 Q–AREA DESIGN CURVE ===")

def Q_vs_A(A_range, U, m_h, cp_h, Th_in, m_c, cp_c, Tc_in, flow="counter"):
    C_h, C_c = m_h*cp_h, m_c*cp_c
    C_min = min(C_h, C_c); Cr = C_min / max(C_h, C_c)
    Qs = []
    for A in A_range:
        NTU = U * A / C_min
        if flow == "counter":
            if Cr >= 0.999:
                eps = NTU / (1 + NTU)
            else:
                eps = (1 - np.exp(-NTU*(1-Cr))) / (1 - Cr*np.exp(-NTU*(1-Cr)))
        else:
            eps = (1 - np.exp(-NTU*(1+Cr))) / (1 + Cr)
        Qs.append(eps * C_min * (Th_in - Tc_in) / 1000)  # kW
    return np.array(Qs)

A_range = np.linspace(5, 80, 50)
fig, ax = plt.subplots(figsize=(9, 5))
for U, style in [(500, "-"), (800, "-"), (1200, "-"), (2000, "--")]:
    Q = Q_vs_A(A_range, U, 2.5, 4180, 120, 3.0, 4180, 25)
    ax.plot(A_range, Q, style, lw=2, label=f"U = {U} W/m²K")
ax.set_xlabel("Heat Transfer Area A [m²]")
ax.set_ylabel("Heat Duty Q [kW]")
ax.set_title("Design Curve: Heat Duty vs Area (counter-flow)")
ax.legend()
ax.axvline(23.56, color="gray", ls=":", alpha=0.7)
ax.text(24.5, 200, "Current Shell-Tube A", fontsize=9, color="gray")
plt.tight_layout()
plt.savefig("15_Q_vs_Area_design_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 15_Q_vs_Area_design_curve.png")


![15_Q_vs_Area_design_curve.png](15_Q_vs_Area_design_curve.png)

*Heat duty vs area design curves at different U.*


In [ ]:
print("=== 22.5 SIMPLE TUBE-SHEET LAYOUT SKETCH ===")

def tube_layout(n_tubes=100, pitch_ratio=1.25, do=0.025, pattern="triangular"):
    """Generate approximate 2-D tube coordinates."""
    pitch = pitch_ratio * do
    coords = []
    # Rough hexagonal / square packing inside a circle
    R_shell = 0.22
    row = 0
    y = -R_shell + pitch/2
    while y < R_shell and len(coords) < n_tubes:
        x_off = (pitch/2 if (row % 2 and pattern == "triangular") else 0)
        x = -R_shell + pitch/2 + x_off
        while x < R_shell and len(coords) < n_tubes:
            if x**2 + y**2 < (R_shell - do/2)**2:
                coords.append((x, y))
            x += pitch
        y += pitch * (np.sqrt(3)/2 if pattern == "triangular" else 1)
        row += 1
    return np.array(coords), R_shell

coords, R = tube_layout(100, 1.25, 0.025, "triangular")
fig, ax = plt.subplots(figsize=(6, 6))
circle = plt.Circle((0, 0), R, fill=False, ec="steelblue", lw=2)
ax.add_patch(circle)
ax.scatter(coords[:,0], coords[:,1], s=28, c="tomato", edgecolors="k", lw=0.4)
ax.set_aspect("equal")
ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
ax.set_title(f"Tube-Sheet Layout Sketch  ({len(coords)} tubes, triangular pitch)")
ax.set_xlim(-R*1.15, R*1.15); ax.set_ylim(-R*1.15, R*1.15)
plt.tight_layout()
plt.savefig("16_tube_sheet_layout.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Placed {len(coords)} tubes. Saved 16_tube_sheet_layout.png")


![16_tube_sheet_layout.png](16_tube_sheet_layout.png)

*Simple triangular-pitch tube-sheet layout sketch.*


In [ ]:
print("=== 22.6 UNIT CONVERTER + RESULTS CSV EXPORT ===")

def convert(value, from_unit, to_unit):
    """Minimal engineering unit converter."""
    factors = {
        ("m2", "ft2"): 10.7639,
        ("ft2", "m2"): 0.092903,
        ("W_m2K", "Btu_hft2F"): 0.1761,
        ("Btu_hft2F", "W_m2K"): 5.678,
        ("kW", "Btu_h"): 3412.14,
        ("Btu_h", "kW"): 0.000293,
        ("kPa", "psi"): 0.145038,
        ("psi", "kPa"): 6.89476,
        ("C", "F"): None,  # special
        ("F", "C"): None,
        ("m2K_W", "ft2hF_Btu"): 5.678,
        ("ft2hF_Btu", "m2K_W"): 0.1761,
    }
    if from_unit == "C" and to_unit == "F":
        return value * 9/5 + 32
    if from_unit == "F" and to_unit == "C":
        return (value - 32) * 5/9
    key = (from_unit, to_unit)
    if key not in factors:
        raise ValueError(f"Unsupported conversion {from_unit} → {to_unit}")
    return value * factors[key]

print("Examples:")
print(f"  23.56 m²  = {convert(23.56, 'm2', 'ft2'):.1f} ft²")
print(f"  800 W/m²K = {convert(800, 'W_m2K', 'Btu_hft2F'):.1f} Btu/h·ft²·°F")
print(f"  120 °C    = {convert(120, 'C', 'F'):.1f} °F")
print(f"  35 kPa    = {convert(35, 'kPa', 'psi'):.2f} psi")

# Export a summary CSV
summary = {
    "metric": ["Q_kW_ShellTube", "Q_kW_Plate", "Q_kW_Finned",
               "A_ShellTube_m2", "U_design_W_m2K", "Rf_crit_m2K_W"],
    "value": [673, 806, 430, 23.56, 800, 1.8e-4]
}
df_sum = pd.DataFrame(summary)
df_sum.to_csv("hx_results_summary.csv", index=False)
print("\nWrote hx_results_summary.csv")
print(df_sum.to_string(index=False))


### v2.2 modules ready

New figures: `12_LMTD_F_chart.png`, `13_axial_temperature_profiles.png`, `14_TEMA_fouling_factors.png`, `15_Q_vs_Area_design_curve.png`, `16_tube_sheet_layout.png`  
New data: `hx_results_summary.csv`


## 23. More Engineering Modules (v2.3)

| Module | Description |
|--------|-------------|
| h & U from correlations | Dittus–Boelter (tube) + simple Kern-style (shell) |
| Closed-loop U | Estimate overall U from geometry + flows instead of assuming it |
| LMTD vs ε-NTU | Side-by-side duty calculation with F-factor |
| Parametric sweep | Vary one input (m, T, U) and plot Q / ε |
| Cleaning strategy | Compare no-clean vs periodic clean on Rf, efficiency, cost |
| Design summary export | Text design sheet for reports |


In [ ]:
print("=== 23.1 HEAT-TRANSFER COEFFICIENTS & CLOSED-LOOP U ===")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def dittus_boelter(Re, Pr, k, di, heating=True):
    """Nu = 0.023 Re^0.8 Pr^n  →  h = Nu * k / di"""
    if Re < 2300:
        # laminar fallback (constant Nu ~ 3.66 for constant wall T)
        Nu = 3.66
    else:
        n = 0.4 if heating else 0.3
        Nu = 0.023 * Re**0.8 * Pr**n
    return Nu * k / di

def kern_shell_h(Re_s, Pr, k, do, mu_w_mu=1.0):
    """Very simplified Kern-style shell-side h."""
    if Re_s <= 0:
        return 50.0
    j_h = 0.36 * Re_s**(-0.55)   # crude Colburn j-factor style
    Nu = j_h * Re_s * Pr**(1/3) * (mu_w_mu)**0.14
    return max(Nu * k / do, 50.0)

def estimate_U_shell_tube(hx, m_h, m_c, rho=980, mu=4e-4, k_f=0.65, cp=4180,
                         Rf_i=0.0, Rf_o=5e-5, n_passes=2):
    """Estimate overall U from geometry + flows (tube = hot side)."""
    di, do, L, n_t = hx.di, hx.do, hx.L, hx.n_tubes
    # Tube-side
    A_flow_t = n_t * np.pi * (di/2)**2 / n_passes
    v_t = m_h / (rho * max(A_flow_t, 1e-8))
    Re_t = rho * v_t * di / mu
    Pr = mu * cp / k_f
    hi = dittus_boelter(Re_t, Pr, k_f, di, heating=False)  # hot fluid cooling
    # Shell-side
    As = hx.Ds * hx.B * 0.5
    v_s = m_c / (rho * max(As, 1e-8))
    Re_s = rho * v_s * do / mu
    ho = kern_shell_h(Re_s, Pr, k_f, do)
    U = hx.overall_U(hi=hi, ho=ho, Rf_i=Rf_i, Rf_o=Rf_o)
    return {
        "hi": hi, "ho": ho, "U": U,
        "Re_t": Re_t, "Re_s": Re_s, "v_t": v_t, "v_s": v_s, "Pr": Pr
    }

est = estimate_U_shell_tube(hx_shell, m_h=2.5, m_c=3.0)
print("Estimated coefficients (Shell-Tube, water-like fluid):")
for k, v in est.items():
    print(f"  {k:6s} = {v:,.2f}")

res_closed = hx_shell.steady_state(2.5, 4180, 120, 3.0, 4180, 25, U=est["U"])
print(f"\nClosed-loop duty: Q = {res_closed['Q']/1000:.1f} kW, ε = {res_closed['eps']:.3f}")


In [ ]:
print("=== 23.2 LMTD METHOD vs ε-NTU ===")

def lmtd_method_Q(m_h, cp_h, Th_in, Th_out, m_c, cp_c, Tc_in, Tc_out, U, A, F=1.0):
    """Q = U A F LMTD  (requires outlet temps)."""
    dT1 = Th_in - Tc_out
    dT2 = Th_out - Tc_in
    LMTD = BaseHeatExchanger.lmtd(dT1, dT2)
    if np.isnan(LMTD):
        return np.nan, LMTD
    return U * A * F * LMTD, LMTD

# Use ε-NTU outlets as the "true" state, then recover Q via LMTD
U_use = est["U"]
r = hx_shell.steady_state(2.5, 4180, 120, 3.0, 4180, 25, U=U_use)
Q_ntu = r["Q"]
Q_lmtd, LMTD = lmtd_method_Q(
    2.5, 4180, 120, r["Th_out"], 3.0, 4180, 25, r["Tc_out"],
    U_use, hx_shell.A, F=1.0
)

print(f"ε-NTU  Q = {Q_ntu/1000:.2f} kW")
print(f"LMTD   Q = {Q_lmtd/1000:.2f} kW   (LMTD = {LMTD:.2f} K, F = 1.0)")
print(f"Relative difference: {100*abs(Q_ntu - Q_lmtd)/Q_ntu:.3f} %")

# Show effect of F < 1 (multipass correction)
print("\nEffect of LMTD correction factor F:")
for F in [1.0, 0.95, 0.90, 0.85, 0.80]:
    Qf, _ = lmtd_method_Q(2.5, 4180, 120, r["Th_out"], 3.0, 4180, 25, r["Tc_out"], U_use, hx_shell.A, F=F)
    print(f"  F = {F:.2f}  →  Q = {Qf/1000:.1f} kW")


In [ ]:
print("=== 23.3 PARAMETRIC SWEEP (Q & ε vs key inputs) ===")

def sweep(param_name, values, base_kwargs=None):
    base = dict(m_h=2.5, cp_h=4180, Th_in=120, m_c=3.0, cp_c=4180, Tc_in=25, U=800)
    if base_kwargs:
        base.update(base_kwargs)
    rows = []
    for v in values:
        kw = base.copy()
        kw[param_name] = v
        try:
            r = hx_shell.steady_state(**kw)
            rows.append({param_name: v, "Q_kW": r["Q"]/1000, "eps": r["eps"], "Th_out": r["Th_out"], "Tc_out": r["Tc_out"]})
        except Exception as e:
            rows.append({param_name: v, "Q_kW": np.nan, "eps": np.nan, "Th_out": np.nan, "Tc_out": np.nan})
    return pd.DataFrame(rows)

m_h_vals = np.linspace(1.0, 5.0, 25)
df_mh = sweep("m_h", m_h_vals)

Th_vals = np.linspace(80, 160, 25)
df_Th = sweep("Th_in", Th_vals)

U_vals = np.linspace(300, 1500, 25)
df_U = sweep("U", U_vals)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
axes[0].plot(df_mh["m_h"], df_mh["Q_kW"], "b-o", ms=3, lw=1.5)
axes[0].set_xlabel("m_h [kg/s]"); axes[0].set_ylabel("Q [kW]"); axes[0].set_title("Duty vs hot flow")

axes[1].plot(df_Th["Th_in"], df_Th["Q_kW"], "r-o", ms=3, lw=1.5)
axes[1].set_xlabel("Th_in [°C]"); axes[1].set_ylabel("Q [kW]"); axes[1].set_title("Duty vs hot inlet T")

axes[2].plot(df_U["U"], df_U["Q_kW"], "g-o", ms=3, lw=1.5)
axes[2].set_xlabel("U [W/m²K]"); axes[2].set_ylabel("Q [kW]"); axes[2].set_title("Duty vs overall U")

plt.suptitle("Parametric Sweeps — Shell-Tube HX", fontweight="bold")
plt.tight_layout()
plt.savefig("17_parametric_sweeps.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 17_parametric_sweeps.png")


In [ ]:
print("=== 23.4 PERIODIC CLEANING STRATEGY COMPARISON ===")

# Simple Kern–Seaton style Rf growth; cleaning resets Rf to near-zero
def simulate_cleaning(days=365, clean_every=None, Rf_inf=1.2e-4, tau_f_days=120,
                      U_clean=950, Q_clean_kW=680, energy_price=0.12, clean_cost=4500):
    t = np.arange(days)
    Rf = np.zeros(days)
    rf = 0.0
    n_cleans = 0
    for d in range(days):
        if clean_every and d > 0 and d % clean_every == 0:
            rf = 1e-6
            n_cleans += 1
        # asymptotic approach
        rf = Rf_inf - (Rf_inf - rf) * np.exp(-1.0 / tau_f_days)
        Rf[d] = rf
    U = 1.0 / (1.0 / U_clean + Rf)
    Q = Q_clean_kW * (U / U_clean)
    lost = (Q_clean_kW - Q) * 24 * energy_price   # $/day
    energy_loss = lost.sum()
    clean_tot = n_cleans * clean_cost
    return {
        "days": t, "Rf": Rf, "Q": Q, "U": U,
        "energy_loss_$": energy_loss,
        "clean_cost_$": clean_tot,
        "total_$": energy_loss + clean_tot,
        "n_cleans": n_cleans
    }

strategies = {
    "No cleaning": simulate_cleaning(clean_every=None),
    "Every 90 d": simulate_cleaning(clean_every=90),
    "Every 60 d": simulate_cleaning(clean_every=60),
    "Every 45 d": simulate_cleaning(clean_every=45),
    "Every 30 d": simulate_cleaning(clean_every=30),
}

print(f"{'Strategy':<14} {'Cleans':>7} {'Energy $':>12} {'Clean $':>10} {'Total $':>10}")
for name, s in strategies.items():
    print(f"{name:<14} {s['n_cleans']:>7} {s['energy_loss_$']:>12,.0f} {s['clean_cost_$']:>10,.0f} {s['total_$']:>10,.0f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, s in strategies.items():
    axes[0].plot(s["days"], s["Q"], lw=1.5, label=name)
axes[0].set_xlabel("Day"); axes[0].set_ylabel("Q [kW]")
axes[0].set_title("Heat duty under cleaning strategies")
axes[0].legend(fontsize=8)

names = list(strategies.keys())
totals = [strategies[n]["total_$"] for n in names]
colors = ["crimson" if t == max(totals) else "seagreen" if t == min(totals) else "steelblue" for t in totals]
axes[1].bar(names, totals, color=colors, edgecolor="k")
axes[1].set_ylabel("Annual cost [USD]")
axes[1].set_title("Energy loss + cleaning cost")
axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig("18_cleaning_strategies.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 18_cleaning_strategies.png")
best = min(strategies.items(), key=lambda kv: kv[1]["total_$"])
print(f"\nLowest total cost under these assumptions: {best[0]} (${best[1]['total_$']:,.0f}/yr)")


In [ ]:
print("=== 23.5 DESIGN SUMMARY TEXT EXPORT ===")

def design_summary(hx, m_h, cp_h, Th_in, m_c, cp_c, Tc_in, U=None, path="hx_design_summary.txt"):
    if U is None:
        est_local = estimate_U_shell_tube(hx, m_h, m_c)
        U = est_local["U"]
        extra = est_local
    else:
        extra = {}
    r = hx.steady_state(m_h, cp_h, Th_in, m_c, cp_c, Tc_in, U=U)
    lines = []
    lines.append("=" * 60)
    lines.append("HEAT EXCHANGER DESIGN SUMMARY")
    lines.append("=" * 60)
    lines.append(f"Unit name          : {hx.name}")
    lines.append(f"Type               : {type(hx).__name__}")
    lines.append(f"Area A             : {r['A']:.2f} m²")
    lines.append(f"Overall U          : {r['U']:.1f} W/m²K")
    lines.append(f"NTU                : {r['NTU']:.2f}")
    lines.append(f"Effectiveness ε    : {r['eps']:.3f}")
    lines.append(f"Heat duty Q        : {r['Q']/1000:.1f} kW")
    lines.append(f"Th_in → Th_out     : {Th_in:.1f} → {r['Th_out']:.1f} °C")
    lines.append(f"Tc_in → Tc_out     : {Tc_in:.1f} → {r['Tc_out']:.1f} °C")
    if extra:
        lines.append("-" * 40)
        lines.append(f"Estimated hi       : {extra.get('hi', float('nan')):.0f} W/m²K")
        lines.append(f"Estimated ho       : {extra.get('ho', float('nan')):.0f} W/m²K")
        lines.append(f"Tube Re            : {extra.get('Re_t', float('nan')):.0f}")
        lines.append(f"Shell Re           : {extra.get('Re_s', float('nan')):.0f}")
    lines.append("=" * 60)
    text = "\n".join(lines)
    with open(path, "w") as f:
        f.write(text + "\n")
    print(text)
    print(f"\nWrote {path}")
    return path

design_summary(hx_shell, 2.5, 4180, 120, 3.0, 4180, 25)


### v2.3 modules ready

New figures: `17_parametric_sweeps.png`, `18_cleaning_strategies.png`  
New files: `hx_design_summary.txt`


## 24. More Engineering Modules (v2.4)

| Module | Description |
|--------|-------------|
| ε–NTU design charts | Effectiveness vs NTU for counter / parallel / cross flow |
| Off-design map | Q contour-style map vs hot flow and inlet temperature |
| Fouling footprint | Extra energy use and CO₂ from degraded U |
| Fixed-duty sizing | Required area for Shell-Tube / Plate / Finned at same Q target |
| Tube vibration screen | Simple critical velocity check (flow-induced vibration) |
| Startup energy | Rough thermal-mass energy to reach steady state |


In [ ]:
print("=== 24.1 ε–NTU DESIGN CHARTS ===")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def eps_ntu_curve(NTU, Cr, flow="counter"):
    NTU = np.asarray(NTU, dtype=float)
    Cr = float(Cr)
    if flow == "counter":
        if Cr >= 0.999:
            return NTU / (1 + NTU)
        return (1 - np.exp(-NTU * (1 - Cr))) / (1 - Cr * np.exp(-NTU * (1 - Cr)))
    if flow == "parallel":
        return (1 - np.exp(-NTU * (1 + Cr))) / (1 + Cr)
    # cross unmixed-unmixed approx
    return 1 - np.exp((1 / max(Cr, 1e-6)) * NTU ** 0.22 * (np.exp(-Cr * NTU ** 0.78) - 1))

NTU = np.linspace(0, 6, 120)
Cr_list = [0.0, 0.25, 0.5, 0.75, 1.0]
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, flow, title in zip(axes, ["counter", "parallel", "cross"],
                           ["Counter-flow", "Parallel-flow", "Cross-flow"]):
    for Cr in Cr_list:
        ax.plot(NTU, eps_ntu_curve(NTU, Cr, flow), lw=1.8, label=f"Cr={Cr}")
    ax.set_xlabel("NTU"); ax.set_title(title); ax.set_ylim(0, 1.05); ax.grid(True, alpha=0.3)
axes[0].set_ylabel("Effectiveness ε"); axes[0].legend(fontsize=8)
plt.suptitle("ε–NTU Design Charts", fontweight="bold")
plt.tight_layout()
plt.savefig("19_eps_NTU_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 19_eps_NTU_charts.png")


In [ ]:
print("=== 24.2 OFF-DESIGN PERFORMANCE MAP ===")

m_vals = np.linspace(1.2, 4.5, 18)
T_vals = np.linspace(90, 150, 18)
Q_map = np.zeros((len(T_vals), len(m_vals)))
for i, Th in enumerate(T_vals):
    for j, mh in enumerate(m_vals):
        try:
            r = hx_shell.steady_state(mh, 4180, Th, 3.0, 4180, 25, U=800)
            Q_map[i, j] = r["Q"] / 1000
        except Exception:
            Q_map[i, j] = np.nan

fig, ax = plt.subplots(figsize=(8, 5.5))
cs = ax.contourf(m_vals, T_vals, Q_map, levels=12, cmap="viridis")
cbar = plt.colorbar(cs, ax=ax)
cbar.set_label("Q [kW]")
ax.contour(m_vals, T_vals, Q_map, levels=8, colors="k", linewidths=0.4, alpha=0.5)
ax.plot(2.5, 120, "r*", ms=14, label="Design point")
ax.set_xlabel("Hot mass flow m_h [kg/s]")
ax.set_ylabel("Hot inlet temperature [°C]")
ax.set_title("Off-Design Performance Map — Shell-Tube (U=800)")
ax.legend(loc="upper left")
plt.tight_layout()
plt.savefig("20_offdesign_map.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 20_offdesign_map.png")


In [ ]:
print("=== 24.3 FOULING ENERGY & CO₂ FOOTPRINT ===")

# Assume lost heat must be made up by a boiler / heater
CO2_factor_kg_per_kWh = 0.45   # grid/fuel mix dependent
ENERGY_PRICE = 0.12
Q_clean = 680.0  # kW

Rf_range = np.linspace(0, 2.5e-4, 40)
U_clean = 950.0
U_f = 1.0 / (1.0 / U_clean + Rf_range)
Q_f = Q_clean * (U_f / U_clean)
lost_kW = Q_clean - Q_f
lost_kWh_day = lost_kW * 24
cost_day = lost_kWh_day * ENERGY_PRICE
co2_day = lost_kWh_day * CO2_factor_kg_per_kWh

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(Rf_range * 1e4, cost_day, "b-", lw=2)
axes[0].set_xlabel("Rf × 10⁴ [m²K/W]")
axes[0].set_ylabel("Extra energy cost [$/day]")
axes[0].set_title("Daily energy cost of fouling")
axes[0].axvline(1.8, color="r", ls="--", alpha=0.7, label="Typical critical Rf")
axes[0].legend()

axes[1].plot(Rf_range * 1e4, co2_day, "g-", lw=2)
axes[1].set_xlabel("Rf × 10⁴ [m²K/W]")
axes[1].set_ylabel("Extra CO₂ [kg/day]")
axes[1].set_title("Daily CO₂ footprint of fouling")
axes[1].axvline(1.8, color="r", ls="--", alpha=0.7)
plt.tight_layout()
plt.savefig("21_fouling_footprint.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 21_fouling_footprint.png")
print(f"At Rf = 1.8e-4: ~${cost_day[np.argmin(np.abs(Rf_range-1.8e-4))]:.0f}/day extra energy, "
      f"~{co2_day[np.argmin(np.abs(Rf_range-1.8e-4))]:.0f} kg CO₂/day")


In [ ]:
print("=== 24.4 FIXED-DUTY SIZING COMPARISON ===")

def area_for_duty(hx_factory, Q_target_kW, m_h, cp_h, Th_in, m_c, cp_c, Tc_in, U, flow="counter"):
    """Find approximate area needed to meet a target duty (scale from a reference geometry)."""
    # Binary search on area scale factor
    Q_target = Q_target_kW * 1000
    lo, hi = 0.2, 5.0
    best_A, best_Q = None, None
    for _ in range(30):
        mid = 0.5 * (lo + hi)
        hx = hx_factory(mid)
        r = hx.steady_state(m_h, cp_h, Th_in, m_c, cp_c, Tc_in, U=U)
        if r["Q"] < Q_target:
            lo = mid
        else:
            hi = mid
        best_A, best_Q = r["A"], r["Q"]
    return best_A, best_Q / 1000

Q_target = 700.0  # kW

def factory_shell(scale):
    return ShellTubeHeatExchanger(n_tubes=int(100*scale), tube_length=3.0*scale**0.5)

def factory_plate(scale):
    return PlateHeatExchanger(n_plates=int(60*scale), plate_area=0.25)

def factory_fin(scale):
    return FinnedTubeHeatExchanger(n_tubes=int(80*scale), L=3.5*scale**0.5)

rows = []
for name, factory, U in [("Shell-Tube", factory_shell, 800),
                          ("Plate", factory_plate, 2400),
                          ("Finned-Tube", factory_fin, 45)]:
    try:
        A, Q = area_for_duty(factory, Q_target, 2.5, 4180, 120, 3.0, 4180, 25, U)
        rows.append({"HX": name, "A_required_m2": A, "Q_achieved_kW": Q, "U_used": U})
    except Exception as e:
        rows.append({"HX": name, "A_required_m2": np.nan, "Q_achieved_kW": np.nan, "U_used": U})

df_size = pd.DataFrame(rows)
print(f"Target duty: {Q_target} kW\n")
print(df_size.round(2).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df_size["HX"], df_size["A_required_m2"], color=["steelblue", "seagreen", "coral"], edgecolor="k")
ax.set_ylabel("Required area [m²]")
ax.set_title(f"Area needed for Q ≈ {Q_target} kW")
plt.tight_layout()
plt.savefig("22_fixed_duty_sizing.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 22_fixed_duty_sizing.png")


In [ ]:
print("=== 24.5 TUBE VIBRATION SCREEN (simplified) ===")

def critical_velocity_screen(do=0.025, L_span=0.3, t_wall=0.0025, rho_tube=7800,
                            E=2.0e11, rho_fluid=980, v_fluid=1.2):
    """
    Very simplified screen:
    - Natural frequency of a simply-supported tube span (beam theory)
    - Vortex-shedding frequency ~ St * v / do  (St ≈ 0.2)
    - Flag if f_shed is within 20% of f_natural (resonance risk)
    """
    # Cross-section properties (thin wall approx)
    di = do - 2 * t_wall
    I = np.pi / 64 * (do**4 - di**4)
    mass_per_len = rho_tube * np.pi / 4 * (do**2 - di**2) + rho_fluid * np.pi / 4 * di**2  # incl. fluid
    # Fundamental frequency simply supported: f = (pi/2) * sqrt(EI/(m L^4))
    f_n = (np.pi / 2) * np.sqrt(E * I / (mass_per_len * L_span**4))
    St = 0.2
    f_shed = St * v_fluid / do
    ratio = f_shed / f_n if f_n > 0 else np.inf
    status = "RISK" if 0.8 <= ratio <= 1.2 else "OK"
    return {"f_natural_Hz": f_n, "f_shed_Hz": f_shed, "ratio": ratio, "status": status,
            "L_span_m": L_span, "v_m_s": v_fluid}

print(f"{'L_span [m]':>10} {'v [m/s]':>8} {'f_n [Hz]':>10} {'f_shed':>10} {'ratio':>8} {'status':>8}")
for Lspan in [0.2, 0.3, 0.45, 0.6]:
    for v in [0.8, 1.2, 1.8]:
        r = critical_velocity_screen(L_span=Lspan, v_fluid=v)
        print(f"{Lspan:10.2f} {v:8.1f} {r['f_natural_Hz']:10.1f} {r['f_shed_Hz']:10.2f} {r['ratio']:8.2f} {r['status']:>8}")

print("\nNote: This is a screening check only — not a full TEMA vibration analysis.")


In [ ]:
print("=== 24.6 STARTUP THERMAL-MASS ENERGY ===")

def startup_energy(M_cp=1.8e6, T_start=25.0, T_ss=95.0, eta_heater=0.9):
    """
    Rough energy to heat metal + fluid hold-up from T_start to steady-state level.
    M_cp in J/K (same order as transient model).
    """
    E_j = M_cp * (T_ss - T_start)
    E_kWh = E_j / 3.6e6
    E_fuel_kWh = E_kWh / eta_heater
    return {"E_stored_kWh": E_kWh, "E_fuel_kWh": E_fuel_kWh, "delta_T": T_ss - T_start}

s = startup_energy()
print(f"Thermal mass energy to raise hold-up by {s['delta_T']:.0f} K:")
print(f"  Stored in HX     : {s['E_stored_kWh']:.1f} kWh")
print(f"  Fuel/electricity : {s['E_fuel_kWh']:.1f} kWh (η={0.9})")
print("Useful for estimating startup cost and time-to-steady with the transient model.")


### v2.4 modules ready

New figures: `19_eps_NTU_charts.png`, `20_offdesign_map.png`, `21_fouling_footprint.png`, `22_fixed_duty_sizing.png`


## 25. More Engineering Modules (v2.5)

| Module | Description |
|--------|-------------|
| Typical U table | Rule-of-thumb overall U by service pair |
| Air-cooled seasonal derate | Ambient temperature effect on air-side duty |
| Approach temperature | Enforce minimum ΔT_approach in design checks |
| 2-stream heat recovery | Simple hot/cold stream matching & target Q |
| KPI JSON export | Single-file snapshot of key performance indicators |
| Life-cycle energy loss | Cumulative lost duty over operating years |


In [ ]:
print("=== 25.1 TYPICAL OVERALL U VALUES (rule of thumb) ===")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json as _json
from datetime import datetime

u_table = [
    ("Water → Water", 800, 1500),
    ("Water → Organic liquid", 300, 700),
    ("Water → Gas", 20, 80),
    ("Steam → Water", 1000, 4000),
    ("Steam → Organic liquid", 300, 1000),
    ("Oil → Oil", 100, 300),
    ("Oil → Water", 200, 500),
    ("Condensing organic → Water", 400, 1000),
    ("Air-cooled (process → air)", 30, 60),
    ("Cryogenic gas → gas", 50, 150),
]
df_u = pd.DataFrame(u_table, columns=["Service", "U_min", "U_max"])
df_u["U_mid"] = 0.5 * (df_u["U_min"] + df_u["U_max"])
print(df_u.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(df_u))
ax.barh(y, df_u["U_max"] - df_u["U_min"], left=df_u["U_min"], color="steelblue", edgecolor="k", height=0.6)
ax.set_yticks(y); ax.set_yticklabels(df_u["Service"])
ax.set_xlabel("Overall U [W/m²K]")
ax.set_title("Typical overall heat-transfer coefficients by service")
plt.tight_layout()
plt.savefig("23_typical_U_values.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 23_typical_U_values.png")


In [ ]:
print("=== 25.2 AIR-COOLED SEASONAL DERATE ===")

# Simple model: air-side driving force shrinks as ambient rises
T_process_in = 120.0
T_air_design = 25.0
Q_design = 430.0  # kW from earlier finned-tube example

T_amb = np.linspace(-5, 45, 51)
# Approximate derate ~ proportional to (T_process - T_amb) / (T_process - T_air_design)
driving = np.maximum(T_process_in - T_amb, 1.0)
driving_des = T_process_in - T_air_design
Q_season = Q_design * (driving / driving_des)
# Cap at ~1.15 of design in very cold weather (fan limits etc.)
Q_season = np.clip(Q_season, 0, Q_design * 1.15)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(T_amb, Q_season, "b-", lw=2, label="Available duty")
ax.axhline(Q_design, color="gray", ls="--", label="Design duty")
ax.axvline(T_air_design, color="orange", ls=":", label="Design ambient")
ax.fill_between(T_amb, Q_season, Q_design, where=Q_season < Q_design, color="crimson", alpha=0.2, label="Derate region")
ax.set_xlabel("Ambient temperature [°C]")
ax.set_ylabel("Heat duty [kW]")
ax.set_title("Air-cooled HX — seasonal capacity derate")
ax.legend()
plt.tight_layout()
plt.savefig("24_aircooled_seasonal_derate.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 24_aircooled_seasonal_derate.png")
print(f"At 40 °C ambient: Q ≈ {Q_season[np.argmin(np.abs(T_amb-40))]:.0f} kW "
      f"({100*Q_season[np.argmin(np.abs(T_amb-40))]/Q_design:.0f}% of design)")


In [ ]:
print("=== 25.3 APPROACH TEMPERATURE CHECK ===")

def approach_check(Th_in, Th_out, Tc_in, Tc_out, min_approach=5.0):
    """Return approach temperatures and pass/fail vs minimum."""
    # Hot outlet vs cold inlet (cold end) and hot inlet vs cold outlet (hot end)
    approach_cold = Th_out - Tc_in
    approach_hot = Th_in - Tc_out
    min_app = min(approach_cold, approach_hot)
    status = "PASS" if min_app >= min_approach else "FAIL"
    return {
        "approach_cold_end": approach_cold,
        "approach_hot_end": approach_hot,
        "min_approach": min_app,
        "required": min_approach,
        "status": status
    }

r = hx_shell.steady_state(2.5, 4180, 120, 3.0, 4180, 25, U=800)
chk = approach_check(120, r["Th_out"], 25, r["Tc_out"], min_approach=5.0)
print(f"Th_out={r['Th_out']:.1f} °C, Tc_out={r['Tc_out']:.1f} °C")
print(f"Cold-end approach : {chk['approach_cold_end']:.1f} K")
print(f"Hot-end approach  : {chk['approach_hot_end']:.1f} K")
print(f"Minimum approach  : {chk['min_approach']:.1f} K  (required ≥ {chk['required']} K) → {chk['status']}")

# Show how approach shrinks as U*A increases (more effective HX)
U_list = np.linspace(200, 2000, 20)
apps = []
for U in U_list:
    rr = hx_shell.steady_state(2.5, 4180, 120, 3.0, 4180, 25, U=U)
    apps.append(approach_check(120, rr["Th_out"], 25, rr["Tc_out"])["min_approach"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(U_list, apps, "b-o", ms=4, lw=1.5)
ax.axhline(5, color="r", ls="--", label="Min approach = 5 K")
ax.set_xlabel("Overall U [W/m²K]")
ax.set_ylabel("Minimum approach [K]")
ax.set_title("Approach temperature vs overall U (fixed geometry)")
ax.legend()
plt.tight_layout()
plt.savefig("25_approach_vs_U.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 25_approach_vs_U.png")


In [ ]:
print("=== 25.4 SIMPLE 2-STREAM HEAT RECOVERY MATCHING ===")

def heat_recovery_target(Th_in, Tc_in, m_h, cp_h, m_c, cp_c, dT_min=10.0):
    """
    Maximum recoverable heat with a given minimum approach (pinch-style).
    Assumes counter-flow limit.
    """
    C_h, C_c = m_h * cp_h, m_c * cp_c
    # Max possible without approach constraint
    Q_max_unlimited = min(C_h, C_c) * (Th_in - Tc_in)
    # With approach: hot can cool to Tc_in + dT_min, cold can heat to Th_in - dT_min
    Q_h_lim = C_h * max(Th_in - (Tc_in + dT_min), 0)
    Q_c_lim = C_c * max((Th_in - dT_min) - Tc_in, 0)
    Q_target = min(Q_h_lim, Q_c_lim, Q_max_unlimited)
    return {
        "Q_target_kW": Q_target / 1000,
        "Q_unlimited_kW": Q_max_unlimited / 1000,
        "recovery_frac": Q_target / Q_max_unlimited if Q_max_unlimited > 0 else 0,
        "dT_min": dT_min
    }

hr = heat_recovery_target(120, 25, 2.5, 4180, 3.0, 4180, dT_min=10)
print(f"Unlimited Q (ε=1)     : {hr['Q_unlimited_kW']:.1f} kW")
print(f"With ΔT_min = 10 K    : {hr['Q_target_kW']:.1f} kW")
print(f"Recoverable fraction  : {100*hr['recovery_frac']:.1f} %")

dTmins = np.linspace(2, 25, 24)
Qs = [heat_recovery_target(120, 25, 2.5, 4180, 3.0, 4180, dT_min=dt)["Q_target_kW"] for dt in dTmins]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dTmins, Qs, "m-", lw=2)
ax.set_xlabel("Minimum approach ΔT_min [K]")
ax.set_ylabel("Max recoverable Q [kW]")
ax.set_title("Heat recovery target vs minimum approach")
plt.tight_layout()
plt.savefig("26_heat_recovery_vs_approach.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 26_heat_recovery_vs_approach.png")


In [ ]:
print("=== 25.5 LIFE-CYCLE ENERGY LOSS & KPI JSON EXPORT ===")

# Cumulative lost duty over N years with gradual fouling (no intermediate clean)
years = np.arange(1, 11)
Rf_inf = 1.5e-4
tau_years = 2.5
U_clean = 950.0
Q_clean = 680.0
hours_per_year = 8000

cum_loss_MWh = []
for y in years:
    # average Rf over the year (approx mid-year value)
    t = y - 0.5
    Rf = Rf_inf * (1 - np.exp(-t / tau_years))
    U = 1.0 / (1.0 / U_clean + Rf)
    Q = Q_clean * (U / U_clean)
    loss_MWh = (Q_clean - Q) * hours_per_year / 1000
    cum_loss_MWh.append(loss_MWh if not cum_loss_MWh else cum_loss_MWh[-1] + loss_MWh)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(years, cum_loss_MWh, "s-", color="crimson", lw=2)
ax.set_xlabel("Operating year")
ax.set_ylabel("Cumulative lost energy [MWh]")
ax.set_title("Life-cycle energy loss from fouling (no intermediate cleaning)")
plt.tight_layout()
plt.savefig("27_lifecycle_energy_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved 27_lifecycle_energy_loss.png")
print(f"10-year cumulative loss ≈ {cum_loss_MWh[-1]:,.0f} MWh")

# KPI JSON snapshot
kpis = {
    "generated": datetime.now().isoformat(timespec="seconds"),
    "unit": "Shell-Tube HX",
    "Q_design_kW": 673,
    "A_m2": 23.56,
    "U_design_W_m2K": 800,
    "eps": 0.68,
    "Rf_crit_m2K_W": 1.8e-4,
    "lifecycle_10y_loss_MWh": round(cum_loss_MWh[-1], 1),
    "typical_cleaning_interval_days": 60,
}
with open("hx_kpis.json", "w") as f:
    _json.dump(kpis, f, indent=2)
print("Wrote hx_kpis.json")
print(_json.dumps(kpis, indent=2))


### v2.5 modules ready

New figures: `23_typical_U_values.png` … `27_lifecycle_energy_loss.png`  
New data: `hx_kpis.json`
